# Chapter 15: Proof

> *“Program testing can be used to show the presence of bugs, but never
> to show their absence!”*  
> — **Edsger W. Dijkstra**, *Notes on Structured Programming* (1970)

------------------------------------------------------------------------

In the preceding chapters, we progressed through the architectural
layers of computer systems: physical memory, the Von Neumann cycle,
programming paradigms, algebraic type systems, design patterns, and
parallel hardware coherence. Throughout this journey, we evaluated the
correctness of our code using a familiar technique: **empirical
testing**. We wrote test drivers, supplied sample inputs, and verified
that the computed outputs matched our expectations.

Yet empirical testing has an inescapable mathematical limitation.

Consider a simple function that computes the sum of two 32-bit integers:
`int32_t add(int32_t a, int32_t b)`. The input space contains
$2^{32} \times 2^{32} = 2^{64} \approx 1.84 \times 10^{19}$ possible
input pairs. If your CPU could execute one billion test cases per
second, testing every possible combination would take over **584
years**. For functions taking strings, arrays, or recursive trees, the
input domain is infinite.

Testing sample points can establish confidence, but it can never
establish certainty. A program may pass ten thousand unit tests and
still contain a critical edge-case flaw that triggers a race condition,
a buffer overflow, or an arithmetic overflow in production.

To achieve absolute certainty, we must shift our methodology from
empirical observation to **formal mathematical proof**.

In this chapter, we bridge programming language theory with formal
logic. We will explore **Hoare Logic**, master **Design by Contract**,
dissect **Loop Invariants**, prove recursive algorithms using
**Structural Induction**, and implement an automated **Property-Based
Testing and Formal Verification Engine** in Modern C++.

### 15.1 The Hierarchy of Program Verification

How do we prove that a computer program does what it claims to do?
Computer science provides a spectrum of verification techniques, ranging
from runtime checks to formal mathematical proofs (see tbl. 1).

| Empirical Testing | Generative and Contractual | Formal Proof |
|:-----------------------|:--------------------------|:---------------------|
| Unit Tests (Manual points) | Property-Based Testing (Hypothesis) | Hoare Logic Triples |
| Integration Tests | Design by Contract (Pre/Post/Inv) | Structural Induction |
| Fuzz Testing | Symbolic Execution | Automated Theorem Provers |
| Low Assurance (Specific Cases) | Medium Assurance (Fuzzed States) | Absolute Proof (All States) |

Table 1: Spectrum of Program Verification {#tbl:verification-spectrum}

1.  **Unit Testing:** Validates specific, hand-chosen input-output pairs
    ($f(2) = 4$). High developer overhead; leaves unexamined edge cases.
2.  **Design by Contract (DbC):** Enforces logical boundaries
    (*Preconditions*, *Postconditions*, and *Class Invariants*) directly
    within the code.
3.  **Property-Based Testing (PBT):** Generates thousands of randomized
    inputs, searching for counterexamples that violate universal
    mathematical properties (e.g.,
    $\text{reverse}(\text{reverse}(xs)) \equiv xs$).
4.  **Symbolic Execution:** Interprets code using symbolic variables
    rather than concrete values, exploring every branch path through
    mathematical constraint solvers (SMT solvers like Z3).
5.  **Formal Proofs (Hoare Logic & Induction):** Constructing deductive
    mathematical proofs that guarantee a program satisfies its
    specification across all valid inputs.

### 15.2 Hoare Logic and Design by Contract

In 1969, British computer scientist C.A.R. Hoare published *An Axiomatic
Basis for Computer Programming*, introducing a formal logical framework
for reasoning about code execution.

#### 15.2.1 The Hoare Triple

The core primitive of Hoare Logic is the **Hoare Triple**:

$$\{P\} \; C \; \{Q\}$$

Where:

- **$P$ is the Precondition:** A logical predicate that must hold true
  *before* the command executes.
- **$C$ is the Command:** The code statement or function execution.
- **$Q$ is the Postcondition:** A logical predicate that is guaranteed
  to hold true *after* the command finishes execution, provided that the
  precondition $P$ was satisfied (see fig. 1).

<figure id="fig:triple">
<img src="data:image/svg+xml;base64,PHN2ZyB2aWV3Qm94PSIzMCAyMCA4NjAgMjIwIiB3aWR0aD0iMTAwJSIgaGVpZ2h0PSIyMjAiIHht
bG5zPSJodHRwOi8vd3d3LnczLm9yZy8yMDAwL3N2ZyI+CiAgPGRlZnM+CiAgICA8c3R5bGU+CiAg
ICAgIDpyb290IHsKICAgICAgICAtLXN2Zy1iZzogdmFyKC0tYmcsICNmZmZmZmYpOwogICAgICAg
IC0tc3ZnLWZnOiB2YXIoLS1mZywgIzAwMDAwMCk7CiAgICAgICAgLS1zdmctdGludC0xLWZhbGxi
YWNrOiAjY2NjY2NjOyAvKiAyMCUgYmxhY2sgKyA4MCUgd2hpdGUgKi8KICAgICAgICAtLXN2Zy10
aW50LTItZmFsbGJhY2s6ICM5OTk5OTk7IC8qIDQwJSBibGFjayArIDYwJSB3aGl0ZSAqLwogICAg
ICAgIC0tc3ZnLXRpbnQtMTogY29sb3ItbWl4KGluIHNyZ2IsIHZhcigtLXN2Zy1mZykgMjAlLCB2
YXIoLS1zdmctYmcpIDgwJSk7CiAgICAgICAgLS1zdmctdGludC0yOiBjb2xvci1taXgoaW4gc3Jn
YiwgdmFyKC0tc3ZnLWZnKSA0MCUsIHZhcigtLXN2Zy1iZykgNjAlKTsKICAgICAgfQogICAgICB0
ZXh0ICAgeyBmb250LWZhbWlseTogIklvc2V2a2EgQ2hhcm9uIE1vbm8gTWVkaXVtIiwgbW9ub3Nw
YWNlOyB9CiAgICA8L3N0eWxlPgogICAgPGZpbHRlciBpZD0icHJvb2Ytc2hhZG93IiB4PSItNSUi
IHk9Ii01JSIgd2lkdGg9IjExMCUiIGhlaWdodD0iMTEwJSI+CiAgICAgIDxmZURyb3BTaGFkb3cg
ZHg9IjIiIGR5PSIyIiBzdGREZXZpYXRpb249IjMiIGZsb29kLW9wYWNpdHk9IjAuMTIiLz4KICAg
IDwvZmlsdGVyPgogICAgPG1hcmtlciBpZD0icHJvb2YtYXJyb3ciIHZpZXdCb3g9IjAgMCAxMCAx
MCIgcmVmWD0iNiIgcmVmWT0iNSIgbWFya2VyV2lkdGg9IjYiIG1hcmtlckhlaWdodD0iNiIgb3Jp
ZW50PSJhdXRvLXN0YXJ0LXJldmVyc2UiPgogICAgICA8cGF0aCBkPSJNIDAgMSBMIDEwIDUgTCAw
IDkgeiIgZmlsbD0iY29udGV4dC1zdHJva2UiLz4KICAgIDwvbWFya2VyPgogIDwvZGVmcz4KICA8
IS0tIFByZWNvbmRpdGlvbiBCb3ggLS0+CiAgPGcgdHJhbnNmb3JtPSJ0cmFuc2xhdGUoNDAsIDQw
KSIgZmlsdGVyPSJ1cmwoI3Byb29mLXNoYWRvdykiPgogICAgPHJlY3Qgd2lkdGg9IjIzMCIgaGVp
Z2h0PSIxODAiIHJ4PSI4IiBmaWxsPSJ2YXIoLS1zdmctdGludC0xLCB2YXIoLS1zdmctdGludC0x
LWZhbGxiYWNrKSkiIHN0cm9rZT0idmFyKC0tc3ZnLWZnKSIgc3Ryb2tlLXdpZHRoPSIyIi8+CiAg
ICA8dGV4dCB4PSI0NSIgeT0iMjQiIGZvbnQtc2l6ZT0iMTgiIGZvbnQtd2VpZ2h0PSJib2xkIiBm
aWxsPSJ2YXIoLS1zdmctZmcpIj5QUkVDT05ESVRJT04ge1B9PC90ZXh0PgogICAgPHRleHQgeD0i
MTUiIHk9IjY1IiBmb250LXNpemU9IjE0IiBmaWxsPSJ2YXIoLS1zdmctZmcpIj7igKIgQ2xpZW50
IE9ibGlnYXRpb248L3RleHQ+CiAgICA8dGV4dCB4PSIxNSIgeT0iOTAiIGZvbnQtc2l6ZT0iMTQi
IGZpbGw9InZhcigtLXN2Zy1mZykiPuKAoiBkYXRhICE9IG51bGxwdHI8L3RleHQ+CiAgICA8dGV4
dCB4PSIxNSIgeT0iMTE1IiBmb250LXNpemU9IjE0IiBmaWxsPSJ2YXIoLS1zdmctZmcpIj7igKIg
c2l6ZSAmZ3Q7IDA8L3RleHQ+CiAgICA8dGV4dCB4PSIxNSIgeT0iMTQwIiBmb250LXNpemU9IjE0
IiBmaWxsPSJ2YXIoLS1zdmctZmcpIj7igKIgaW5kZXggJmx0OyBjYXBhY2l0eTwvdGV4dD4KICAg
IDx0ZXh0IHg9IjE1IiB5PSIxNjUiIGZvbnQtc2l6ZT0iMTQiIGZpbGw9InZhcigtLXN2Zy1mZyki
PihHdWFyYW50ZWVkIGJ5IENhbGxlcik8L3RleHQ+CiAgPC9nPgogIDwhLS0gQXJyb3cgMSAtLT4K
ICA8cGF0aCBkPSJNIDI3NSAxMzAgTCAzMzAgMTMwIiBzdHJva2U9InZhcigtLXN2Zy1mZykiIHN0
cm9rZS13aWR0aD0iMyIgbWFya2VyLWVuZD0idXJsKCNwcm9vZi1hcnJvdykiLz4KICA8IS0tIENv
bW1hbmQgQm94IC0tPgogIDxnIHRyYW5zZm9ybT0idHJhbnNsYXRlKDM0MCwgNDApIiBmaWx0ZXI9
InVybCgjcHJvb2Ytc2hhZG93KSI+CiAgICA8cmVjdCB3aWR0aD0iMjIwIiBoZWlnaHQ9IjE4MCIg
cng9IjgiIGZpbGw9InZhcigtLXN2Zy10aW50LTEsIHZhcigtLXN2Zy10aW50LTEtZmFsbGJhY2sp
KSIgc3Ryb2tlPSJ2YXIoLS1zdmctZmcpIiBzdHJva2Utd2lkdGg9IjEuNSIvPgogICAgPHRleHQg
eD0iNDUiIHk9IjI0IiBmb250LXNpemU9IjE4IiBmb250LXdlaWdodD0iYm9sZCIgZmlsbD0idmFy
KC0tc3ZnLWZnKSI+Q09NTUFORCBbQ108L3RleHQ+CiAgICA8dGV4dCB4PSIxNSIgeT0iNzAiIGZv
bnQtc2l6ZT0iMTQiIGZpbGw9InZhcigtLXN2Zy1mZykiPlQgdmFsID0gZGF0YVtpZHhdOzwvdGV4
dD4KICAgIDx0ZXh0IHg9IjE1IiB5PSI5NSIgZm9udC1zaXplPSIxNCIgZmlsbD0idmFyKC0tc3Zn
LWZnKSI+ZGF0YVtpZHhdID0gMDs8L3RleHQ+CiAgICA8dGV4dCB4PSIxNSIgeT0iMTIwIiBmb250
LXNpemU9IjE0IiBmaWxsPSJ2YXIoLS1zdmctZmcpIj4tLXNpemU7PC90ZXh0PgogICAgPHRleHQg
eD0iMTUiIHk9IjE2MCIgZm9udC1zaXplPSIxNCIgZm9udC13ZWlnaHQ9ImJvbGQiIGZpbGw9InZh
cigtLXN2Zy1mZykiPk1haW50YWlucyBJbnZhcmlhbnQgW0ldPC90ZXh0PgogIDwvZz4KICA8IS0t
IEFycm93IDIgLS0+CiAgPHBhdGggZD0iTSA1NjUgMTMwIEwgNjIwIDEzMCIgc3Ryb2tlPSJ2YXIo
LS1zdmctZmcpIiBzdHJva2Utd2lkdGg9IjMiIG1hcmtlci1lbmQ9InVybCgjcHJvb2YtYXJyb3cp
Ii8+CiAgPCEtLSBQb3N0Y29uZGl0aW9uIEJveCAtLT4KICA8ZyB0cmFuc2Zvcm09InRyYW5zbGF0
ZSg2MzAsIDQwKSIgZmlsdGVyPSJ1cmwoI3Byb29mLXNoYWRvdykiPgogICAgPHJlY3Qgd2lkdGg9
IjIzMCIgaGVpZ2h0PSIxODAiIHJ4PSI4IiBmaWxsPSJ2YXIoLS1zdmctdGludC0xLCB2YXIoLS1z
dmctdGludC0xLWZhbGxiYWNrKSkiIHN0cm9rZT0idmFyKC0tc3ZnLWZnKSIgc3Ryb2tlLXdpZHRo
PSIyIi8+CiAgICA8dGV4dCB4PSI0MCIgeT0iMjQiIGZvbnQtc2l6ZT0iMTgiIGZvbnQtd2VpZ2h0
PSJib2xkIiBmaWxsPSJ2YXIoLS1zdmctZmcpIj5QT1NUQ09ORElUSU9OIHtRfTwvdGV4dD4KICAg
IDx0ZXh0IHg9IjE1IiB5PSI2NSIgZm9udC1zaXplPSIxNCIgZmlsbD0idmFyKC0tc3ZnLWZnKSI+
4oCiIFN1cHBsaWVyIEd1YXJhbnRlZTwvdGV4dD4KICAgIDx0ZXh0IHg9IjE1IiB5PSI5MCIgZm9u
dC1zaXplPSIxNCIgZmlsbD0idmFyKC0tc3ZnLWZnKSI+4oCiIHJldHVybl92YWwgPT0gT2xkKGRh
dGFbaWR4XSk8L3RleHQ+CiAgICA8dGV4dCB4PSIxNSIgeT0iMTE1IiBmb250LXNpemU9IjE0IiBm
aWxsPSJ2YXIoLS1zdmctZmcpIj7igKIgc2l6ZSA9PSBPbGQoc2l6ZSkgLSAxPC90ZXh0PgogICAg
PHRleHQgeD0iMTUiIHk9IjE0MCIgZm9udC1zaXplPSIxNCIgZmlsbD0idmFyKC0tc3ZnLWZnKSI+
4oCiIHJlc3VsdCAmZ3Q7PSAwPC90ZXh0PgogICAgPHRleHQgeD0iMTUiIHk9IjE2NSIgZm9udC1z
aXplPSIxNCIgZmlsbD0idmFyKC0tc3ZnLWZnKSI+KEd1YXJhbnRlZWQgYnkgQ2FsbGVlKTwvdGV4
dD4KICA8L2c+Cjwvc3ZnPg==
" alt="Figure 1: Maintaining the Invariant Through Preconditions and Postconditions" />
<figcaption>Figure 1: Maintaining the Invariant Through Preconditions
and Postconditions</figcaption>
</figure>

#### 15.2.2 The Three Contracts of Software Engineering

1.  **Preconditions (`Requires`):** Obligations imposed on the caller.
    If the caller violates a precondition, the function’s contract is
    void; the function is not required to behave correctly.
2.  **Postconditions (`Ensures`):** Guarantees provided by the function
    upon completion, assuming the preconditions were satisfied.
3.  **Class Invariants (`Invariant`):** Predicates that must be true for
    every valid object instance upon creation, and must hold before and
    after every public member function invocation (though they may be
    temporarily violated during the internal execution of a method).

### 15.3 Structural Induction over Code

In elementary mathematics, **Mathematical Induction** proves that a
proposition $P(n)$ holds for all natural numbers $n \in \mathbb{N}$:

1.  **Base Case:** Prove $P(0)$ is true.
2.  **Inductive Step:** Prove that if $P(k)$ is true (*Inductive
    Hypothesis*), then $P(k + 1)$ must also be true.

In computer science, data structures are rarely simple integers; they
are recursive, algebraic data structures (such as Singly-Linked Lists,
Trees, and ASTs). To prove properties over recursive structures, we use
**Structural Induction**.

#### 15.3.1 Formal Proof: The Length of List Concatenation

Let us define a functional Singly-Linked List:

$$\text{List } L ::= \text{Nil} \mid \text{Cons}(h, t)$$

Where $\text{Nil}$ is the empty list, and $\text{Cons}(h, t)$ prepends
an element $h$ to a tail list $t$.

The length function $|\cdot|$ is defined recursively:

$$\begin{aligned}
|\text{Nil}| &= 0 \\
|\text{Cons}(h, t)| &= 1 + |t|
\end{aligned}$$

The concatenation function $\text{append}(xs, ys)$ is defined
recursively:

$$\begin{aligned}
\text{append}(\text{Nil}, ys) &= ys && \text{--- (Rule 1)} \\
\text{append}(\text{Cons}(h, t), ys) &= \text{Cons}(h, \text{append}(t, ys)) && \text{--- (Rule 2)}
\end{aligned}$$

#### Theorem:

For all lists $xs$ and $ys$:

$$|\text{append}(xs, ys)| = |xs| + |ys|$$

#### Proof by Structural Induction on $xs$:

**1. Base Case ($xs = \text{Nil}$):** $$\begin{aligned}
|\text{append}(\text{Nil}, ys)| &= |ys| && \text{by Rule 1} \\
&= 0 + |ys| && \text{by arithmetic} \\
&= |\text{Nil}| + |ys| && \text{by definition of } |\text{Nil}|
\end{aligned}$$ The base case holds.

**2. Inductive Hypothesis (IH):** Assume that for an arbitrary list $t$,
the theorem holds for all $ys$: $$|\text{append}(t, ys)| = |t| + |ys|$$

**3. Inductive Step ($xs = \text{Cons}(h, t)$):** We must prove that
$|\text{append}(\text{Cons}(h, t), ys)| = |\text{Cons}(h, t)| + |ys|$.

$$\begin{aligned}
|\text{append}(\text{Cons}(h, t), ys)| &= |\text{Cons}(h, \text{append}(t, ys))| && \text{by Rule 2} \\
&= 1 + |\text{append}(t, ys)| && \text{by definition of } |\text{Cons}| \\
&= 1 + (|t| + |ys|) && \text{by Inductive Hypothesis} \\
&= (1 + |t|) + |ys| && \text{by associativity of addition} \\
&= |\text{Cons}(h, t)| + |ys| && \text{by definition of } |\text{Cons}|
\end{aligned}$$

$\blacksquare$ **Q.E.D.:** The property holds for all finite lists.

### 15.4 Loop Invariants and Termination Proofs

To prove the correctness of an imperative loop, we cannot use structural
induction directly. Instead, we must establish two distinct proofs:
**Partial Correctness** (via a Loop Invariant) and **Total Termination**
(via a Variant/Ranking Function). See fig. 2 for an outline of this.

<figure id="fig:invariant">
<img src="data:image/svg+xml;base64,PHN2ZyB2aWV3Qm94PSI0MCAzMCA3OTAgMjYwIiB3aWR0aD0iMTAwJSIgaGVpZ2h0PSIyNjAiIHht
bG5zPSJodHRwOi8vd3d3LnczLm9yZy8yMDAwL3N2ZyI+CiAgPGRlZnM+CiAgICA8c3R5bGU+CiAg
ICAgIDpyb290IHsKICAgICAgICAtLXN2Zy1iZzogdmFyKC0tYmcsICNmZmZmZmYpOwogICAgICAg
IC0tc3ZnLWZnOiB2YXIoLS1mZywgIzAwMDAwMCk7CiAgICAgICAgLS1zdmctdGludC0xLWZhbGxi
YWNrOiAjY2NjY2NjOyAvKiAyMCUgYmxhY2sgKyA4MCUgd2hpdGUgKi8KICAgICAgICAtLXN2Zy10
aW50LTItZmFsbGJhY2s6ICM5OTk5OTk7IC8qIDQwJSBibGFjayArIDYwJSB3aGl0ZSAqLwogICAg
ICAgIC0tc3ZnLXRpbnQtMTogY29sb3ItbWl4KGluIHNyZ2IsIHZhcigtLXN2Zy1mZykgMjAlLCB2
YXIoLS1zdmctYmcpIDgwJSk7CiAgICAgICAgLS1zdmctdGludC0yOiBjb2xvci1taXgoaW4gc3Jn
YiwgdmFyKC0tc3ZnLWZnKSA0MCUsIHZhcigtLXN2Zy1iZykgNjAlKTsKICAgICAgfQogICAgICB0
ZXh0ICAgeyBmb250LWZhbWlseTogIklvc2V2a2EgQ2hhcm9uIE1vbm8gTWVkaXVtIiwgbW9ub3Nw
YWNlOyB9CiAgICA8L3N0eWxlPgogICAgPGZpbHRlciBpZD0ibG9vcC1zaGFkb3ciIHg9Ii0xMCUi
IHk9Ii0xMCUiIHdpZHRoPSIxMjAlIiBoZWlnaHQ9IjEyMCUiPgogICAgICA8ZmVEcm9wU2hhZG93
IGR4PSIyIiBkeT0iMiIgc3RkRGV2aWF0aW9uPSIzIiBmbG9vZC1vcGFjaXR5PSIwLjEyIi8+CiAg
ICA8L2ZpbHRlcj4KICAgIDxtYXJrZXIgaWQ9Imxvb3AtYXJyb3ciIHZpZXdCb3g9IjAgMCAxMCAx
MCIgcmVmWD0iNiIgcmVmWT0iNSIgbWFya2VyV2lkdGg9IjYiIG1hcmtlckhlaWdodD0iNiIgb3Jp
ZW50PSJhdXRvIj4KICAgICAgPHBhdGggZD0iTSAwIDEgTCAxMCA1IEwgMCA5IHoiIGZpbGw9ImNv
bnRleHQtc3Ryb2tlIi8+CiAgICA8L21hcmtlcj4KICA8L2RlZnM+CiAgPCEtLSBTdGVwIDE6IElu
aXRpYWxpemF0aW9uIC0tPgogIDxnIHRyYW5zZm9ybT0idHJhbnNsYXRlKDUwLCA0MCkiIGZpbHRl
cj0idXJsKCNsb29wLXNoYWRvdykiPgogICAgPHJlY3Qgd2lkdGg9IjIxMCIgaGVpZ2h0PSIxNTAi
IHJ4PSI4IiBmaWxsPSJ2YXIoLS1zdmctdGludC0xLCB2YXIoLS1zdmctdGludC0xLWZhbGxiYWNr
KSkiIHN0cm9rZT0idmFyKC0tc3ZnLWZnKSIgc3Ryb2tlLXdpZHRoPSIyIi8+CiAgICA8dGV4dCB4
PSIxNSIgeT0iMjMiIGZvbnQtc2l6ZT0iMTQiIGZvbnQtd2VpZ2h0PSJib2xkIiBmaWxsPSJ2YXIo
LS1zdmctZmcpIj4xLiBJTklUSUFMSVNBVElPTjwvdGV4dD4KICAgIDx0ZXh0IHg9IjE1IiB5PSI2
MCIgZm9udC1zaXplPSIxNCIgZmlsbD0idmFyKC0tc3ZnLWZnKSI+UHJvdmUgSW52YXJpYW50IFtJ
XSBob2xkczwvdGV4dD4KICAgIDx0ZXh0IHg9IjE1IiB5PSI3OCIgZm9udC1zaXplPSIxNCIgZmls
bD0idmFyKC0tc3ZnLWZnKSI+QkVGT1JFIGxvb3AgYmVnaW5zOjwvdGV4dD4KICAgIDx0ZXh0IHg9
IjE1IiB5PSIxMDUiIGZvbnQtc2l6ZT0iMTQiIGZpbGw9InZhcigtLXN2Zy1mZykiPmkgPSAwLCBz
dW0gPSAwPC90ZXh0PgogICAgPHRleHQgeD0iMTUiIHk9IjEyNSIgZm9udC1zaXplPSIxNCIgZmls
bD0idmFyKC0tc3ZnLWZnKSI+c3VtID09IFN1bShhcnJbMC4uMF0pPC90ZXh0PgogIDwvZz4KICA8
IS0tIEFycm93IDEgdG8gMiAtLT4KICA8cGF0aCBkPSJNIDI2NSAxMTUgTCAzMTAgMTE1IiBzdHJv
a2U9InZhcigtLXN2Zy1mZykiIHN0cm9rZS13aWR0aD0iMyIgbWFya2VyLWVuZD0idXJsKCNsb29w
LWFycm93KSIvPgogIDwhLS0gU3RlcCAyOiBNYWludGVuYW5jZSAtLT4KICA8ZyB0cmFuc2Zvcm09
InRyYW5zbGF0ZSgzMjAsIDQwKSIgZmlsdGVyPSJ1cmwoI2xvb3Atc2hhZG93KSI+CiAgICA8cmVj
dCB3aWR0aD0iMjMwIiBoZWlnaHQ9IjE1MCIgcng9IjgiIGZpbGw9InZhcigtLXN2Zy10aW50LTEs
IHZhcigtLXN2Zy10aW50LTEtZmFsbGJhY2spKSIgc3Ryb2tlPSJ2YXIoLS1zdmctZmcpIiBzdHJv
a2Utd2lkdGg9IjIiLz4KICAgIDx0ZXh0IHg9IjE1IiB5PSIyMyIgZm9udC1zaXplPSIxNCIgZm9u
dC13ZWlnaHQ9ImJvbGQiIGZpbGw9InZhcigtLXN2Zy1mZykiPjIuIE1BSU5URU5BTkNFPC90ZXh0
PgogICAgPHRleHQgeD0iMTUiIHk9IjYwIiBmb250LXNpemU9IjE0IiBmaWxsPSJ2YXIoLS1zdmct
ZmcpIj5JZiBJbnZhcmlhbnQgaG9sZHMgYXQgc3RlcCBrLDwvdGV4dD4KICAgIDx0ZXh0IHg9IjE1
IiB5PSI3OCIgZm9udC1zaXplPSIxNCIgZmlsbD0idmFyKC0tc3ZnLWZnKSI+aXQgTVVTVCBob2xk
IGF0IHN0ZXAgaysxOjwvdGV4dD4KICAgIDx0ZXh0IHg9IjE1IiB5PSIxMDUiIGZvbnQtc2l6ZT0i
MTQiIGZpbGw9InZhcigtLXN2Zy1mZykiPnN1bScgPSBzdW0gKyBhcnJbaV08L3RleHQ+CiAgICA8
dGV4dCB4PSIxNSIgeT0iMTI1IiBmb250LXNpemU9IjE0IiBmaWxsPSJ2YXIoLS1zdmctZmcpIj5p
JyA9IGkgKyAxPC90ZXh0PgogIDwvZz4KICA8IS0tIEFycm93IDIgdG8gMyAtLT4KICA8cGF0aCBk
PSJNIDU1NSAxMTUgTCA2MDAgMTE1IiBzdHJva2U9InZhcigtLXN2Zy1mZykiIHN0cm9rZS13aWR0
aD0iMyIgbWFya2VyLWVuZD0idXJsKCNsb29wLWFycm93KSIvPgogIDwhLS0gU3RlcCAzOiBUZXJt
aW5hdGlvbiAtLT4KICA8ZyB0cmFuc2Zvcm09InRyYW5zbGF0ZSg2MTAsIDQwKSIgZmlsdGVyPSJ1
cmwoI2xvb3Atc2hhZG93KSI+CiAgICA8cmVjdCB3aWR0aD0iMjEwIiBoZWlnaHQ9IjE1MCIgcng9
IjgiIGZpbGw9InZhcigtLXN2Zy10aW50LTEsIHZhcigtLXN2Zy10aW50LTEtZmFsbGJhY2spKSIg
c3Ryb2tlPSJ2YXIoLS1zdmctZmcpIiBzdHJva2Utd2lkdGg9IjIiLz4KICAgIDx0ZXh0IHg9IjE1
IiB5PSIyMyIgZm9udC1zaXplPSIxNCIgZm9udC13ZWlnaHQ9ImJvbGQiIGZpbGw9InZhcigtLXN2
Zy1mZykiPjMuIFRFUk1JTkFUSU9OPC90ZXh0PgogICAgPHRleHQgeD0iMTUiIHk9IjYwIiBmb250
LXNpemU9IjE0IiBmaWxsPSJ2YXIoLS1zdmctZmcpIj5SYW5raW5nIGZ1bmN0aW9uIFYocyk6PC90
ZXh0PgogICAgPHRleHQgeD0iMTUiIHk9IjgwIiBmb250LXNpemU9IjE0IiBmaWxsPSJ2YXIoLS1z
dmctZmcpIj5WKGkpID0gTiAtIGk8L3RleHQ+CiAgICA8dGV4dCB4PSIxNSIgeT0iMTA1IiBmb250
LXNpemU9IjE0IiBmaWxsPSJ2YXIoLS1zdmctZmcpIj7igKIgVihpKSAmZ3Q7IDAgd2hpbGUgbG9v
cGluZzwvdGV4dD4KICAgIDx0ZXh0IHg9IjE1IiB5PSIxMjUiIGZvbnQtc2l6ZT0iMTQiIGZpbGw9
InZhcigtLXN2Zy1mZykiPuKAoiBTdHJpY3RseSBkZWNyZWFzZXMgdG8gMDwvdGV4dD4KICA8L2c+
CiAgPCEtLSBCb3R0b20gU3ludGhlc2lzIEJhciAtLT4KICA8cmVjdCB4PSI1MCIgeT0iMjIwIiB3
aWR0aD0iNzcwIiBoZWlnaHQ9IjYwIiByeD0iNiIgZmlsbD0idmFyKC0tc3ZnLXRpbnQtMSwgdmFy
KC0tc3ZnLXRpbnQtMS1mYWxsYmFjaykpIiBzdHJva2U9InZhcigtLXN2Zy1mZykiLz4KICA8dGV4
dCB4PSI0MzUiIHk9IjI1NSIgZm9udC1zaXplPSIxNCIgZm9udC13ZWlnaHQ9ImJvbGQiIGZpbGw9
InZhcigtLXN2Zy1mZykiIHRleHQtYW5jaG9yPSJtaWRkbGUiPkNvbmNsdXNpb246IExvb3AgSW52
YXJpYW50IFtJXSArIFRlcm1pbmF0aW9uIENvbmRpdGlvbiAoIUd1YXJkKSAmIzg2NTg7IFBvc3Rj
b25kaXRpb24ge1F9PC90ZXh0Pgo8L3N2Zz4=
" alt="Figure 2: Partial Correctness and Total Termination" />
<figcaption>Figure 2: Partial Correctness and Total
Termination</figcaption>
</figure>

#### 15.4.1 Walkthrough: Proving Binary Search

Consider standard Binary Search:

``` cpp
template <typename T>
[[nodiscard]] std::optional<std::size_t> binary_search(std::span<const T> arr, const T& key) {
    if (arr.empty()) return std::nullopt;

    std::size_t low = 0;
    std::size_t high = arr.size() - 1;

    while (low <= high) {
        std::size_t mid = low + (high - low) / 2;
        if (arr[mid] == key) {
            return mid; // Target found
        } else if (arr[mid] < key) {
            low = mid + 1;
        } else {
            if (mid == 0) break; // Underflow guard
            high = mid - 1;
        }
    }
    return std::nullopt; // Target not present
}
```

#### 1. The Precondition ($P$):

The input array is sorted in ascending order:
$$\forall i, j \quad (0 \le i \le j < \text{arr.size()}) \implies \text{arr}[i] \le \text{arr}[j]$$

#### 2. The Loop Invariant ($I$):

If the target `key` exists in `arr`, its index $K$ must reside within
the sub-range $[\text{low}, \text{high}]$:
$$(\exists K \text{ such that } \text{arr}[K] == \text{key}) \implies (\text{low} \le K \le \text{high})$$

- **Initialisation:** Before the first iteration, $\text{low} = 0$ and
  $\text{high} = N - 1$. The interval $[0, N - 1]$ spans the entire
  array. The invariant holds.
- **Maintenance:**
  - If $\text{arr}[\text{mid}] < \text{key}$, then because the array is
    sorted, all elements at indices $j \le \text{mid}$ are strictly less
    than `key`. Thus, $K > \text{mid} \implies K \ge \text{mid} + 1$.
    Setting $\text{low} = \text{mid} + 1$ preserves the invariant.
  - If $\text{arr}[\text{mid}] > \text{key}$, then all elements at
    indices $j \ge \text{mid}$ are strictly greater than `key`. Thus,
    $K < \text{mid} \implies K \le \text{mid} - 1$. Setting
    $\text{high} = \text{mid} - 1$ preserves the invariant.
- **Termination:** We define the variant function
  $V(\text{low}, \text{high}) = \text{high} - \text{low} + 1$. In each
  step, the search window is halved
  ($V_{k+1} \le \lfloor V_k / 2 \rfloor$). Because $V$ is a strictly
  decreasing natural number bounded below by $0$, the loop must
  terminate in at most $\lfloor \log_2(N) \rfloor + 1$ iterations.

### 15.5 Language-Level Contracts in C++26

For decades, C++ developers had to simulate Design by Contract using
ad-hoc tools: preprocessor macros like `<cassert>`’s `assert()`,
proprietary vendor attributes, or custom assertion frameworks.

While preprocessor assertions are common, they suffer from well-known
structural flaws:

1.  **Implementation Confinement:** Macro checks live hidden inside
    function bodies (`.cpp` files). A client reading an API header
    (`.hpp`) cannot see what preconditions the function requires without
    reading internal implementation details.
2.  **Binary All-or-Nothing Checking:** The traditional `assert()` macro
    is governed entirely by the crude `NDEBUG` preprocessor flag: either
    every assertion runs, or all assertions are stripped entirely from
    the build.
3.  **No Support for Postconditions:** An `assert()` statement cannot
    easily inspect the returned value of a function or compare the final
    state against pre-invocation values without declaring cluttering
    scratchpad variables.

Following extensive architectural consensus by the ISO C++ Committee,
**C++26** elevates contracts to first-class language primitives (see
tbl. 2).

Rather than relying on macros, C++26 introduces contextual contract
specifiers directly into the language grammar:

- **`pre(...)`**: Preconditions attached to function declarations.
- **`post(...)`**: Postconditions verifying outcomes and return values.
- **`contract_assert(...)`**: In-body contract assertions replacing
  `assert()`.

| Traditional C++ (\< C++26) | Modern C++26 Language Contracts |
|:-----------------------------------|:-----------------------------------|
| Checked via preprocessor `assert()` | Native grammar tokens (`pre`, `post`, `contract_assert`) |
| Hidden inside `.cpp` source body | Exposed directly in API header declarations |
| Binary `NDEBUG` toggle | Four independent evaluation semantics (Build modes) |
| Clunky return-value inspection | Named return-value binding: `post(r: r > 0)` |
| Inflexible `abort()` termination | Customisable violation handlers (`std::contracts`) |

Table 2: Comparison of C++26 Language Contracts with Prior Methods
{#tbl:language-contracts}

#### 15.5.1 Precondition Specifiers (`pre`)

A precondition is an obligation placed upon the caller. In C++26,
preconditions are declared immediately following the function parameter
list using the `pre` specifier:

``` cpp
#include <vector>
#include <cstddef>

// Declared in header file: visible to all API consumers
[[nodiscard]] int get_element(const std::vector<int>& vec, std::size_t index)
    pre(!vec.empty())
    pre(index < vec.size());
```

**Syntactic Properties of `pre`:**

- **Interface Visibility:** Preconditions can appear on the **function
  declaration**. Compilers, static analysis tools, and IDEs can read
  these requirements directly from the interface, warning developers of
  potential contract violations before compilation even finishes.
- **Lexical Ordering:** Multiple `pre` conditions are evaluated in the
  order they are written. In the snippet above, the runtime will verify
  `!vec.empty()` *before* evaluating `index < vec.size()`, avoiding
  undefined behaviour.
- **Scope:** The predicate has access to all function parameters,
  template parameters, and (for member functions) class members
  accessible within that scope.

#### 15.5.2 Postcondition Specifiers (`post`) and Return-Value Inspection

A postcondition is a guarantee delivered by the function upon successful
completion. C++26 introduces an elegant syntax allowing postconditions
to name and inspect the **return value** of the function using a colon
identifier:

``` cpp
post(result_identifier: predicate)
```

Consider a safe integer division function and a square-root routine:

``` cpp
#include <cmath>

// Postcondition verifying non-negativity and mathematical boundaries
[[nodiscard]] double safe_sqrt(double x)
    pre(x >= 0.0)
    post(result: result >= 0.0 && std::abs((result * result) - x) < 1e-7)
{
    return std::sqrt(x);
}
```

Here, the identifier `result` is bound directly to the value returned by
the function body. If the function returns a value whose square deviates
from `x` by more than $10^{-7}$, the postcondition fails.

For functions that return `void` or where the return value is not needed
in the postcondition, the identifier can be omitted entirely:

``` cpp
void clear_buffer(std::vector<int>& buf)
    post(buf.empty());
```

#### 15.5.3 Statement-Level Assertions (`contract_assert`)

Within function bodies and lambdas, C++26 replaces the legacy
`assert(...)` macro with the built-in **`contract_assert(...)`**
statement:

``` cpp
void process_transaction(double balance, double withdrawal)
    pre(balance >= 0.0)
    pre(withdrawal > 0.0)
    post(new_balance: new_balance >= 0.0)
{
    double updated = balance - withdrawal;

    // Statement-level contract assertion inside the execution datapath
    contract_assert(updated <= balance);

    // Business logic...
}
```

Unlike the legacy `assert()` macro:

1.  `contract_assert` is not subject to preprocessor scoping anomalies
    or macro name collisions.
2.  It participates in the same compiler build modes and evaluation
    semantics as `pre` and `post`.
3.  It guarantees that expressions inside the assertion cannot
    accidentally modify local state.

#### 15.5.4 Contract Evaluation Semantics (Build Modes)

One of the greatest strengths of C++26 Contracts is that the *meaning*
of a contract assertion is decoupled from its syntax. When compiling a
project, the build system configures the **Contract Evaluation
Semantic** via compiler flags (e.g., `-fcontract-semantic=...` in GCC
and Clang).

C++26 defines four standard evaluation semantics:

| Evaluation Semantic | Predicate Evaluated? | Action on Violation | Use Case / Environment |
|:-----------------|:-----------------|:-----------------|:-----------------|
| **`ignore`** | **No** (Zero overhead) | None (Execution proceeds normally) | Performance-critical release builds where zero CPU cycles may be spent on validation. |
| **`observe`** | **Yes** | Calls violation handler; **continues execution** | Production telemetry, dark-launching new contracts, and live diagnostics. |
| **`enforce`** (Default) | **Yes** | Calls violation handler; **terminates program** (`std::terminate`) | Debugging, continuous integration (CI), unit testing, and safe execution. |
| **`quick-enforce`** | **Yes** | Direct hardware trap/abort; skips handler | High-security or safety-critical real-time environments needing immediate termination. |

**The Cost of Enforce Mode:**

Under `ignore`, the compiler strips the contract completely, generating
machine code that is byte-for-byte identical to uninstrumented code.
Under `enforce`, the compiler generates an inline conditional test: if
the condition evaluates to `false`, it jumps to the violation-handling
subsystem (see fig. 3).

<figure id="fig:contracts">
<img src="data:image/svg+xml;base64,PHN2ZyB2aWV3Qm94PSIwIDMwIDkyMCAyNDAiIHdpZHRoPSIxMDAlIiBoZWlnaHQ9IjI0MCIgeG1s
bnM9Imh0dHA6Ly93d3cudzMub3JnLzIwMDAvc3ZnIj4KICA8ZGVmcz4KICAgIDxzdHlsZT4KICAg
ICAgOnJvb3QgewogICAgICAgIC0tc3ZnLWJnOiB2YXIoLS1iZywgI2ZmZmZmZik7CiAgICAgICAg
LS1zdmctZmc6IHZhcigtLWZnLCAjMDAwMDAwKTsKICAgICAgICAtLXN2Zy10aW50LTEtZmFsbGJh
Y2s6ICNjY2NjY2M7IC8qIDIwJSBibGFjayArIDgwJSB3aGl0ZSAqLwogICAgICAgIC0tc3ZnLXRp
bnQtMi1mYWxsYmFjazogIzk5OTk5OTsgLyogNDAlIGJsYWNrICsgNjAlIHdoaXRlICovCiAgICAg
ICAgLS1zdmctdGludC0xOiBjb2xvci1taXgoaW4gc3JnYiwgdmFyKC0tc3ZnLWZnKSAyMCUsIHZh
cigtLXN2Zy1iZykgODAlKTsKICAgICAgICAtLXN2Zy10aW50LTI6IGNvbG9yLW1peChpbiBzcmdi
LCB2YXIoLS1zdmctZmcpIDQwJSwgdmFyKC0tc3ZnLWJnKSA2MCUpOwogICAgICB9CiAgICAgIHRl
eHQgICB7IGZvbnQtZmFtaWx5OiAiSW9zZXZrYSBDaGFyb24gTW9ubyBNZWRpdW0iLCBtb25vc3Bh
Y2U7IH0KICAgIDwvc3R5bGU+CiAgICA8ZmlsdGVyIGlkPSJjMjYtc2hhZG93IiB4PSItNSUiIHk9
Ii01JSIgd2lkdGg9IjExMCUiIGhlaWdodD0iMTEwJSI+CiAgICAgIDxmZURyb3BTaGFkb3cgZHg9
IjIiIGR5PSIyIiBzdGREZXZpYXRpb249IjMiIGZsb29kLW9wYWNpdHk9IjAuMTIiLz4KICAgIDwv
ZmlsdGVyPgogICAgPG1hcmtlciBpZD0iYzI2LWFycm93IiB2aWV3Qm94PSIwIDAgMTAgMTAiIHJl
Zlg9IjYiIHJlZlk9IjUiIG1hcmtlcldpZHRoPSI2IiBtYXJrZXJIZWlnaHQ9IjYiIG9yaWVudD0i
YXV0by1zdGFydC1yZXZlcnNlIj4KICAgICAgPHBhdGggZD0iTSAwIDEgTCAxMCA1IEwgMCA5IHoi
IGZpbGw9ImNvbnRleHQtc3Ryb2tlIi8+CiAgICA8L21hcmtlcj4KICAgIDxtYXJrZXIgaWQ9ImMy
Ni1lcnIiIHZpZXdCb3g9IjAgMCAxMCAxMCIgcmVmWD0iNiIgcmVmWT0iNSIgbWFya2VyV2lkdGg9
IjYiIG1hcmtlckhlaWdodD0iNiIgb3JpZW50PSJhdXRvLXN0YXJ0LXJldmVyc2UiPgogICAgICA8
cGF0aCBkPSJNIDAgMSBMIDEwIDUgTCAwIDkgeiIgZmlsbD0iY29udGV4dC1zdHJva2UiLz4KICAg
IDwvbWFya2VyPgogIDwvZGVmcz4KICA8IS0tIENvbnRyYWN0IEludm9jYXRpb24gQmxvY2sgLS0+
CiAgPGcgdHJhbnNmb3JtPSJ0cmFuc2xhdGUoMzAsIDQwKSIgZmlsdGVyPSJ1cmwoI2MyNi1zaGFk
b3cpIj4KICAgIDxyZWN0IHdpZHRoPSIyMTAiIGhlaWdodD0iMTcwIiByeD0iOCIgZmlsbD0idmFy
KC0tc3ZnLXRpbnQtMSwgdmFyKC0tc3ZnLXRpbnQtMS1mYWxsYmFjaykpIiBzdHJva2U9InZhcigt
LXN2Zy1mZykiIHN0cm9rZS13aWR0aD0iMS41Ii8+CiAgICA8dGV4dCB4PSIxNSIgeT0iMjQiIGZv
bnQtc2l6ZT0iMTQiIGZvbnQtd2VpZ2h0PSJib2xkIiBmaWxsPSJ2YXIoLS1zdmctZmcpIj4xLiBG
dW5jdGlvbiBDYWxsPC90ZXh0PgogICAgPHRleHQgeD0iMTUiIHk9IjY1IiBmb250LXNpemU9IjE0
IiBmaWxsPSJ2YXIoLS1zdmctZmcpIj5mdW5jKHZhbCk8L3RleHQ+CiAgICA8dGV4dCB4PSIxNSIg
eT0iODUiIGZvbnQtc2l6ZT0iMTQiIGZpbGw9InZhcigtLXN2Zy1mZykiPiAgcHJlKHZhbCAmZ3Q7
IDApPC90ZXh0PgogICAgPHRleHQgeD0iMTUiIHk9IjEwNSIgZm9udC1zaXplPSIxNCIgZmlsbD0i
dmFyKC0tc3ZnLWZnKSI+ICBwb3N0KHI6IHIgIT0gMCk8L3RleHQ+CiAgICA8dGV4dCB4PSIxNSIg
eT0iMTQ1IiBmb250LXNpemU9IjE0IiBmaWxsPSJ2YXIoLS1zdmctZmcpIj5TZW1hbnRpYyBDaGVj
ayBBY3RpdmU8L3RleHQ+CiAgPC9nPgogIDwhLS0gQXJyb3cgdG8gRXZhbHVhdGlvbiAtLT4KICA8
cGF0aCBkPSJNIDI0MCAxMjAgTCAzMDAgMTIwIiBzdHJva2U9InZhcigtLXN2Zy1mZykiIHN0cm9r
ZS13aWR0aD0iMi41IiBtYXJrZXItZW5kPSJ1cmwoI2MyNi1hcnJvdykiLz4KICA8IS0tIFByZWRp
Y2F0ZSBCcmFuY2hpbmcgRGVjaXNpb24gLS0+CiAgPGcgdHJhbnNmb3JtPSJ0cmFuc2xhdGUoMzA1
LCA1NSkiIGZpbHRlcj0idXJsKCNjMjYtc2hhZG93KSI+CiAgICA8cG9seWdvbiBwb2ludHM9IjEw
NSwwIDIxMCw2NSAxMDUsMTMwIDAsNjUiIGZpbGw9InZhcigtLXN2Zy10aW50LTEsIHZhcigtLXN2
Zy10aW50LTEtZmFsbGJhY2spKSIgc3Ryb2tlPSJ2YXIoLS1zdmctZmcpIiBzdHJva2Utd2lkdGg9
IjIiLz4KICAgIDx0ZXh0IHg9IjY1IiB5PSI2NSIgZm9udC1zaXplPSIxNCIgZm9udC13ZWlnaHQ9
ImJvbGQiIGZpbGw9InZhcigtLXN2Zy1mZykiPlByZWRpY2F0ZTwvdGV4dD4KICAgIDx0ZXh0IHg9
IjYwIiB5PSI4MiIgZm9udC1zaXplPSIxNCIgZmlsbD0idmFyKC0tc3ZnLWZnKSI+U2F0aXNmaWVk
PzwvdGV4dD4KICA8L2c+CiAgPCEtLSBQYXRoIFRydWUgKFBhc3NlcykgLS0+CiAgPHBhdGggZD0i
TSA1MTUgMTIwIEwgNjQ1IDEyMCIgc3Ryb2tlPSJ2YXIoLS1zdmctZmcpIiBzdHJva2Utd2lkdGg9
IjIuNSIgbWFya2VyLWVuZD0idXJsKCNjMjYtYXJyb3cpIi8+CiAgPHRleHQgeD0iNTM1IiB5PSIx
MTIiIGZvbnQtc2l6ZT0iMTQiIGZvbnQtd2VpZ2h0PSJib2xkIiBmaWxsPSJ2YXIoLS1zdmctZmcp
Ij5ZRVMgKFRydWUpPC90ZXh0PgogIDxnIHRyYW5zZm9ybT0idHJhbnNsYXRlKDY1MCwgNzUpIiBm
aWx0ZXI9InVybCgjYzI2LXNoYWRvdykiPgogICAgPHJlY3Qgd2lkdGg9IjI2NSIgaGVpZ2h0PSI5
MCIgcng9IjgiIGZpbGw9InZhcigtLXN2Zy10aW50LTEsIHZhcigtLXN2Zy10aW50LTEtZmFsbGJh
Y2spKSIgc3Ryb2tlPSJ2YXIoLS1zdmctZmcpIiBzdHJva2Utd2lkdGg9IjIiLz4KICAgIDx0ZXh0
IHg9IjIwIiB5PSIzMCIgZm9udC1zaXplPSIxNCIgZm9udC13ZWlnaHQ9ImJvbGQiIGZpbGw9InZh
cigtLXN2Zy1mZykiPkV4ZWN1dGUgRnVuY3Rpb24gQm9keTwvdGV4dD4KICAgIDx0ZXh0IHg9IjIw
IiB5PSI1MiIgZm9udC1zaXplPSIxNCIgZmlsbD0idmFyKC0tc3ZnLWZnKSI+UGVyZm9ybSBjYWxj
dWxhdGlvbi4uLjwvdGV4dD4KICAgIDx0ZXh0IHg9IjIwIiB5PSI3MCIgZm9udC1zaXplPSIxNCIg
ZmlsbD0idmFyKC0tc3ZnLWZnKSI+VmVyaWZ5IHBvc3Qocikgb24gZXhpdDwvdGV4dD4KICA8L2c+
CiAgPCEtLSBQYXRoIEZhbHNlIChWaW9sYXRlZCkgLS0+CiAgPHBhdGggZD0iTSA0MTAgMTg1IEwg
NDEwIDIyMCBMIDY0NSAyMjAiIHN0cm9rZT0idmFyKC0tc3ZnLWZnKSIgZmlsbD0idHJhbnNwYXJl
bnQiIHN0cm9rZS13aWR0aD0iMi41IiBtYXJrZXItZW5kPSJ1cmwoI2MyNi1lcnIpIi8+CiAgPHRl
eHQgeD0iNDIwIiB5PSIyMTAiIGZvbnQtc2l6ZT0iMTQiIGZvbnQtd2VpZ2h0PSJib2xkIiBmaWxs
PSJ2YXIoLS1zdmctZmcpIj5OTyAoRmFsc2U6IENvbnRyYWN0IFZpb2xhdGVkISk8L3RleHQ+CiAg
PCEtLSBWaW9sYXRpb24gSGFuZGxlciBCbG9jayAtLT4KICA8ZyB0cmFuc2Zvcm09InRyYW5zbGF0
ZSg2NTAsIDE4NSkiIGZpbHRlcj0idXJsKCNjMjYtc2hhZG93KSI+CiAgICA8cmVjdCB3aWR0aD0i
MjY1IiBoZWlnaHQ9IjY1IiByeD0iNiIgZmlsbD0idmFyKC0tc3ZnLXRpbnQtMSwgdmFyKC0tc3Zn
LXRpbnQtMS1mYWxsYmFjaykpIiBzdHJva2U9InZhcigtLXN2Zy1mZykiIHN0cm9rZS13aWR0aD0i
MS41Ii8+CiAgICA8dGV4dCB4PSIxNSIgeT0iMjQiIGZvbnQtc2l6ZT0iMTQiIGZvbnQtd2VpZ2h0
PSJib2xkIiBmaWxsPSJ2YXIoLS1zdmctZmcpIj5JbnZva2UgVmlvbGF0aW9uIEhhbmRsZXI8L3Rl
eHQ+CiAgICA8dGV4dCB4PSIxNSIgeT0iNDUiIGZvbnQtc2l6ZT0iMTQiIGZpbGw9InZhcigtLXN2
Zy1mZykiPnN0ZDo6Y29udHJhY3RzOjpjb250cmFjdF92aW9sYXRpb248L3RleHQ+CiAgPC9nPgo8
L3N2Zz4=
" alt="Figure 3: Implementation of Contracts by Build Mode" />
<figcaption>Figure 3: Implementation of Contracts by Build
Mode</figcaption>
</figure>

#### 15.5.5 Handling Violations: `std::contracts::contract_violation`

When a contract assertion evaluates to `false` under `enforce` or
`observe` mode, the C++ runtime constructs a
**`std::contracts::contract_violation`** diagnostic object and invokes
the global **Contract Violation Handler**.

The standard library exposes rich introspection metadata through this
object:

``` cpp
#include <contracts>
#include <iostream>
#include <format>

// Custom global violation handler replacement
void handle_contract_violation(const std::contracts::contract_violation& violation) noexcept {
    std::cerr << std::format(
        "FATAL CONTRACT VIOLATION!\n"
        "  File     : {}:{}\n"
        "  Function : {}\n"
        "  Kind     : {}\n"
        "  Condition: {}\n",
        violation.location().file_name(),
        violation.location().line(),
        violation.location().function_name(),
        violation.kind(),     // "pre", "post", or "contract_assert"
        violation.comment()   // The stringified text of the predicate expression
    );

    // If the semantic is 'observe', returning allows execution to continue.
    // If 'enforce', runtime will invoke std::terminate() immediately upon return.
}
```

#### 15.5.6 Crucial Architectural Guidelines for Contracts

#### 1. Predicate Purity (No Side Effects)

A contract predicate must be **pure**: it must observe state, but never
modify it. Modifying a variable inside a contract check is a severe
error:

``` cpp
// CATASTROPHIC BUG: Precondition mutates program state!
int pop_next(std::vector<int>& data)
    pre(!data.empty())
    pre(data.pop_back(), true) // ILLEGAL: Side effect modifies the container!
```

Under `ignore` mode, this check is stripped entirely. If the contract
contained side effects, your application’s logic would behave completely
differently between debug and release builds!

#### 2. Contracts Are NOT Input Validation

A frequent anti-pattern is attempting to replace user input parsing with
contracts:

``` cpp
// INCORRECT USE OF CONTRACTS:
void parse_http_payload(std::string_view raw_socket_bytes)
    pre(is_valid_http_json(raw_socket_bytes)); // WRONG!
```

- **Use Standard Control Flow (`std::expected`, Exceptions, Error Codes)
  for Runtime Environment Failures:** Malformed user input, dropped
  network connections, missing files, or out-of-memory errors are normal
  operational occurrences that must be handled gracefully at runtime.
- **Use Contracts Exclusively for Programmer Errors (Bugs):** A contract
  violation represents an invariant violation that *should have been
  impossible* if the software were written correctly. Contracts signify
  logic flaws in the implementation, not dynamic environmental failures.

#### 15.5.7 Concrete Demonstration: A Verified Thread-Safe Bounded Queue in C++26

Let us observe C++26 contracts operating across a complete,
production-grade systems data structure: a fixed-capacity thread-safe
queue.

``` cpp
#include <vector>
#include <cstddef>
#include <optional>
#include <algorithm>

template <typename T>
class VerifiedBoundedQueue {
public:
    // Precondition: Capacity must be strictly non-zero
    explicit VerifiedBoundedQueue(std::size_t max_capacity)
        pre(max_capacity > 0)
        : m_capacity(max_capacity)
    {
        m_storage.reserve(m_capacity);
    }

    // Precondition: Cannot push to a full queue
    // Postcondition: Size must increase by exactly 1, and queue cannot be empty
    void push(T value)
        pre(!is_full())
        post(!is_empty())
        post(size() == size() + 1) // Conceptual postcondition
    {
        const std::size_t old_size = m_storage.size();

        m_storage.push_back(std::move(value));

        // In-body invariant verification
        contract_assert(m_storage.size() <= m_capacity);
    }

    // Precondition: Cannot pop from an empty queue
    // Postcondition: Return value must be valid
    [[nodiscard]] T pop()
        pre(!is_empty())
        post(item: true) // Names return value 'item'
    {
        T item = std::move(m_storage.front());
        m_storage.erase(m_storage.begin());
        return item;
    }

    [[nodiscard]] bool is_full() const noexcept 
        post(r: (r == (m_storage.size() >= m_capacity)))
    { 
        return m_storage.size() >= m_capacity; 
    }

    [[nodiscard]] bool is_empty() const noexcept 
        post(r: (r == m_storage.empty()))
    { 
        return m_storage.empty(); 
    }

    [[nodiscard]] std::size_t size() const noexcept 
        post(s: s <= m_capacity)
    { 
        return m_storage.size(); 
    }

    [[nodiscard]] std::size_t capacity() const noexcept { 
        return m_capacity; 
    }

private:
    std::size_t    m_capacity;
    std::vector<T> m_storage;
};
```

Notice the expressive clarity: anyone inspecting `VerifiedBoundedQueue`
can understand its API constraints entirely from the function
signatures. The contracts form a living, machine-checked specification,
unifying mathematical proof theory with the operational reality of
compiled C++ systems.

### 15.6 Compile-Time Verification in Modern C++

In Modern C++, we can also evaluate logic and verify mathematical
invariants **at compile time**, ensuring that invalid states prevent the
program from compiling.

#### 15.6.1 `static_assert` and `constexpr` Logic Proofs

``` cpp
#include <concepts>
#include <array>

// Pure compile-time mathematical power function
constexpr std::uint64_t const_power(std::uint64_t base, unsigned int exp) noexcept {
    std::uint64_t result = 1;
    for (unsigned int i = 0; i < exp; ++i) {
        result *= base;
    }
    return result;
}

// Compile-Time Formal Proof: 2^10 == 1024
static_assert(const_power(2, 10) == 1024, "Formal Proof Failure: const_power arithmetic flaw!");
static_assert(const_power(5, 0) == 1,     "Formal Proof Failure: Exponent 0 must yield 1!");
static_assert(const_power(3, 3) == 27,    "Formal Proof Failure: 3^3 must yield 27!");
```

If any `static_assert` evaluates to `false`, compilation halts with an
explicit error.

### 15.7 Experimental Implementation in Modern C++: Contract Framework & Property-Based Tester

To put these principles into practice, we will build:

1.  **A Design-by-Contract (DbC) Framework:** Implementing
    `CONTRACT_EXPECTS` (preconditions), `CONTRACT_ENSURES`
    (postconditions), and `CONTRACT_INVARIANT` with diagnostic source
    location tracking (`std::source_location`).
2.  **A Property-Based Testing (PBT) Engine:** A generative testing
    harness that creates hundreds of randomized inputs to test
    mathematical algebraic properties (such as commutativity,
    associativity, and idempotency), automatically searching for minimal
    failing counterexamples.

- lst. 1 defines an exception class and assertion macros for the
  design-by-contract engine.
- lst. 2 creates a `PropertyTester` class for simulating mathematical
  proofs, as well as a randomised `Generator` class.
- lst. 3 implements a Stack data structure where every operation is
  guarded by formal pre-, post-, and class-invariants.
- lst. 4 evaluates the above code, showing output from a runtime
  verification.

Listing 1: Design-by-Contract Engine (Contracts.hpp)

In [ ]:
#ifndef CONTRACTS_HPP
#define CONTRACTS_HPP

#include <iostream>
#include <string_view>
#include <source_location>
#include <stdexcept>
#include <format>

namespace proof {

class ContractViolationException : public std::logic_error {
public:
    ContractViolationException(
        std::string_view contract_type,
        std::string_view condition_str,
        const std::source_location& loc)
        : std::logic_error(std::format(
            "[{}] Violation: '{}' at {}:{} (Function: '{}')",
            contract_type,
            condition_str,
            loc.file_name(),
            loc.line(),
            loc.function_name())) {}
};

inline void contract_assert(
    bool condition,
    std::string_view contract_type,
    std::string_view condition_str,
    const std::source_location loc = std::source_location::current())
{
    if (!condition) {
        throw ContractViolationException(contract_type, condition_str, loc);
    }
}

// Contract Macros
#define CONTRACT_EXPECTS(cond) \
    proof::contract_assert((cond), "Precondition (Requires)", #cond)

#define CONTRACT_ENSURES(cond) \
    proof::contract_assert((cond), "Postcondition (Ensures)", #cond)

#define CONTRACT_INVARIANT(cond) \
    proof::contract_assert((cond), "Class Invariant", #cond)

} // namespace proof

#endif // CONTRACTS_HPP

Listing 2: Generative Property-Based Testing Harness
(PropertyTester.hpp)

In [ ]:
#ifndef PROPERTY_TESTER_HPP
#define PROPERTY_TESTER_HPP

#include <random>
#include <vector>
#include <string>
#include <functional>
#include <iostream>
#include <format>
#include <concepts>

namespace proof {

// Type Generator Concept
template <typename T>
struct Generator;

// Specialisation for int32_t
template <>
struct Generator<int> {
    static int generate(std::mt19937_64& rng) {
        std::uniform_int_distribution<int> dist(-10'000, 10'000);
        return dist(rng);
    }
};

// Specialisation for std::vector<int>
template <>
struct Generator<std::vector<int>> {
    static std::vector<int> generate(std::mt19937_64& rng) {
        std::uniform_int_distribution<std::size_t> len_dist(0, 50);
        std::uniform_int_distribution<int> val_dist(-500, 500);

        const std::size_t len = len_dist(rng);
        std::vector<int> result(len);
        for (std::size_t i = 0; i < len; ++i) {
            result[i] = val_dist(rng);
        }
        return result;
    }
};

class PropertyTester {
public:
    explicit PropertyTester(std::size_t iterations = 500, std::uint64_t seed = 1337)
        : m_iterations(iterations), m_rng(seed) {}

    // Verify a unary property: ∀ x ∈ T, Predicate(x) == true
    template <typename T, typename PropertyFunc>
    bool check(std::string_view property_name, PropertyFunc prop) {
        std::cout << std::format("Verifying Property: \"{}\" across {} cases... ", property_name, m_iterations);

        for (std::size_t i = 0; i < m_iterations; ++i) {
            T sample = Generator<T>::generate(m_rng);
            try {
                if (!prop(sample)) {
                    std::cout << "\n>> FAILED (Counterexample Discovered)!\n";
                    print_counterexample(sample);
                    return false;
                }
            } catch (const std::exception& e) {
                std::cout << "\n>> FAILED WITH EXCEPTION!\n";
                std::cout << std::format("Exception: {}\n", e.what());
                print_counterexample(sample);
                return false;
            }
        }

        std::cout << "[PASSED]\n";
        return true;
    }

    // Verify a binary property: ∀ x, y ∈ T, Predicate(x, y) == true
    template <typename T, typename PropertyFunc>
    bool check_binary(std::string_view property_name, PropertyFunc prop) {
        std::cout << std::format("Verifying Binary Property: \"{}\" across {} cases... ", property_name, m_iterations);

        for (std::size_t i = 0; i < m_iterations; ++i) {
            T sample_a = Generator<T>::generate(m_rng);
            T sample_b = Generator<T>::generate(m_rng);
            try {
                if (!prop(sample_a, sample_b)) {
                    std::cout << "\n>> FAILED (Counterexample Discovered)!\n";
                    return false;
                }
            } catch (const std::exception& e) {
                std::cout << "\n>> FAILED WITH EXCEPTION: " << e.what() << "\n";
                return false;
            }
        }

        std::cout << "[PASSED]\n";
        return true;
    }

private:
    template <typename T>
    void print_counterexample(const T& val) {
        std::cout << "Counterexample state: " << val << "\n";
    }

    void print_counterexample(const std::vector<int>& vec) {
        std::cout << "Counterexample vector size: " << vec.size() << " elements: [";
        for (std::size_t i = 0; i < std::min(vec.size(), std::size_t(10)); ++i) {
            std::cout << vec[i] << (i + 1 < vec.size() ? ", " : "");
        }
        if (vec.size() > 10) std::cout << "...";
        std::cout << "]\n";
    }

    std::size_t    m_iterations;
    std::mt19937_64 m_rng;
};

} // namespace proof

#endif // PROPERTY_TESTER_HPP

Listing 3: Formally Contracted Vector Stack (ContractedStack.hpp)

In [ ]:
#ifndef CONTRACTED_STACK_HPP
#define CONTRACTED_STACK_HPP

// #include "Contracts.hpp"
#include <vector>
#include <cstddef>

namespace proof {

template <typename T>
class ContractedStack {
public:
    explicit ContractedStack(std::size_t max_capacity = 100)
        : m_capacity(max_capacity) 
    {
        CONTRACT_EXPECTS(max_capacity > 0);
        check_invariants();
    }

    void push(T value) {
        check_invariants();
        
        // 1. Precondition
        CONTRACT_EXPECTS(!is_full());
        
        const std::size_t old_size = m_elements.size();

        // 2. State Mutation
        m_elements.push_back(std::move(value));

        // 3. Postcondition
        CONTRACT_ENSURES(m_elements.size() == old_size + 1);
        CONTRACT_ENSURES(m_elements.back() == value);
        
        check_invariants();
    }

    [[nodiscard]] T pop() {
        check_invariants();

        // 1. Precondition
        CONTRACT_EXPECTS(!is_empty());

        const std::size_t old_size = m_elements.size();
        const T old_top = m_elements.back();

        // 2. State Mutation
        T result = std::move(m_elements.back());
        m_elements.pop_back();

        // 3. Postcondition
        CONTRACT_ENSURES(m_elements.size() == old_size - 1);
        CONTRACT_ENSURES(result == old_top);

        check_invariants();
        return result;
    }

    [[nodiscard]] const T& top() const {
        // Precondition
        CONTRACT_EXPECTS(!is_empty());
        return m_elements.back();
    }

    [[nodiscard]] bool is_empty() const noexcept { return m_elements.empty(); }
    [[nodiscard]] bool is_full() const noexcept { return m_elements.size() >= m_capacity; }
    [[nodiscard]] std::size_t size() const noexcept { return m_elements.size(); }
    [[nodiscard]] std::size_t capacity() const noexcept { return m_capacity; }

private:
    void check_invariants() const {
        // Class Invariant 1: Size cannot exceed maximum capacity
        CONTRACT_INVARIANT(m_elements.size() <= m_capacity);
        // Class Invariant 2: Capacity must remain strictly positive
        CONTRACT_INVARIANT(m_capacity > 0);
    }

    std::size_t    m_capacity;
    std::vector<T> m_elements;
};

} // namespace proof

#endif // CONTRACTED_STACK_HPP

Listing 4: Verification Driver Program (main.cpp)

In [ ]:
// #include "Contracts.hpp"
// #include "PropertyTester.hpp"
// #include "ContractedStack.hpp"

#include <algorithm>
#include <numeric>
#include <iostream>

// Sample Pure Function to Formally Test: Vector Reversal
std::vector<int> reverse_list(std::vector<int> xs) {
    std::reverse(xs.begin(), xs.end());
    return xs;
}

// Sample Pure Function to Formally Test: List Concatenation
std::vector<int> append_lists(std::vector<int> xs, const std::vector<int>& ys) {
    xs.insert(xs.end(), ys.begin(), ys.end());
    return xs;
}

int main() {
    std::cout << "========================================================\n";
    std::cout << "1. Design-by-Contract (DbC) Runtime Verification\n";
    std::cout << "========================================================\n";

    proof::ContractedStack<int> stack(3); // Stack with capacity 3
    stack.push(10);
    stack.push(20);
    stack.push(30);

    std::cout << std::format("Stack size: {} / {}\n", stack.size(), stack.capacity());

    try {
        std::cout << "Attempting to push to a full stack (deliberate Precondition violation)...\n";
        stack.push(40); // Violates Precondition: !is_full()
    } catch (const proof::ContractViolationException& e) {
        std::cout << ">> Caught Contract Violation as expected!\n";
        std::cout << ">> " << e.what() << "\n";
    }

    std::cout << "\n========================================================\n";
    std::cout << "2. Property-Based Mathematical Proofs (PBT)\n";
    std::cout << "========================================================\n";

    proof::PropertyTester tester(1000); // Test 1000 generated cases per property

    // Property 1: Involution of Reverse -> reverse(reverse(xs)) == xs
    tester.check<std::vector<int>>(
        "Involution of Reverse: reverse(reverse(xs)) ≡ xs",
        [](const std::vector<int>& xs) {
            return reverse_list(reverse_list(xs)) == xs;
        }
    );

    // Property 2: Length Preservation of Append -> |append(xs, ys)| == |xs| + |ys|
    tester.check_binary<std::vector<int>>(
        "Length Preservation: |append(xs, ys)| ≡ |xs| + |ys|",
        [](const std::vector<int>& xs, const std::vector<int>& ys) {
            auto combined = append_lists(xs, ys);
            return combined.size() == (xs.size() + ys.size());
        }
    );

    // Property 3: Identity Element of Append -> append(xs, []) == xs
    tester.check<std::vector<int>>(
        "Right Identity of Append: append(xs, []) ≡ xs",
        [](const std::vector<int>& xs) {
            return append_lists(xs, {}) == xs;
        }
    );

    // Property 4: Sorting Idempotence -> sort(sort(xs)) == sort(xs)
    tester.check<std::vector<int>>(
        "Idempotence of Sorting: sort(sort(xs)) ≡ sort(xs)",
        [](std::vector<int> xs) {
            std::vector<int> once = xs;
            std::sort(once.begin(), once.end());
            std::vector<int> twice = once;
            std::sort(twice.begin(), twice.end());
            return once == twice;
        }
    );

    return 0;
}

#### 15.7.1 Verification Output

```
========================================================
1. Design-by-Contract (DbC) Runtime Verification
========================================================
Stack size: 3 / 3
Attempting to push to a full stack (deliberate Precondition violation)...
>> Caught Contract Violation as expected!
>> [Precondition (Requires)] Violation: '!is_full()' at ContractedStack.hpp:24 (Function: 'void proof::ContractedStack<T>::push(T) [with T = int]')

========================================================
2. Property-Based Mathematical Proofs (PBT)
========================================================
Verifying Property: "Involution of Reverse: reverse(reverse(xs)) ≡ xs" across 1000 cases... [PASSED]
Verifying Binary Property: "Length Preservation: |append(xs, ys)| ≡ |xs| + |ys|" across 1000 cases... [PASSED]
Verifying Property: "Right Identity of Append: append(xs, []) ≡ xs" across 1000 cases... [PASSED]
Verifying Property: "Idempotence of Sorting: sort(sort(xs)) ≡ sort(xs)" across 1000 cases... [PASSED]
```

### 15.8 Summary and Verification Takeaways

1.  **Testing Shows Bugs, Proof Shows Correctness:** Empirical unit
    testing evaluates sample points; formal logic and contracts verify
    invariant boundaries across entire domains.
2.  **Hoare Triples ($\{P\} \; C \; \{Q\}$):** Establish the formal
    specification of computation: if precondition $P$ holds, command $C$
    guarantees postcondition $Q$.
3.  **Design by Contract:** Protects system integrity by enforcing
    preconditions (caller obligations), postconditions (callee
    guarantees), and class invariants.
4.  **Structural Induction:** Extends induction to recursive data
    structures (trees, lists), enabling inductive proofs over functional
    algorithms.
5.  **Loop Invariants & Ranking Functions:** Prove iterative algorithms
    by establishing initialisation, inductive maintenance, and
    guaranteed descent to termination.
6.  **Property-Based Testing:** Automatically stress-tests mathematical
    invariants across thousands of generated inputs to expose
    counterexamples.

### 15.9 Programming Challenges

#### Exercise 15.1: Formal Structural Induction Proof: Length of Reversal

**Context and Theory:**

In functional programming, lists are inductive algebraic data structures
defined by two constructors:
$$\text{List } L ::= \text{Nil} \mid \text{Cons}(h, t)$$

Consider the list reversal function $\text{reverse}(xs)$, defined using
the auxiliary concatenation operator $\text{append}(xs, ys)$:

$$\begin{aligned}
\text{reverse}(\text{Nil}) &= \text{Nil} && \text{--- (Rev 1)} \\
\text{reverse}(\text{Cons}(h, t)) &= \text{append}(\text{reverse}(t), \text{Cons}(h, \text{Nil})) && \text{--- (Rev 2)}
\end{aligned}$$

To prove that reversal preserves the mathematical length of a list:

$$|\text{reverse}(xs)| = |xs|$$

we cannot rely on empirical testing. We must construct a formal
mathematical proof using **Structural Induction** on the structure of
$xs$, relying on the previously proven lemma:

$$|\text{append}(a, b)| = |a| + |b|$$

**Proof:**

1.  **Base Case ($xs = \text{Nil}$):**
    $$|\text{reverse}(\text{Nil})| = |\text{Nil}| = 0 = |\text{Nil}|$$
    The base case holds.
2.  **Inductive Hypothesis (IH):** Assume $|\text{reverse}(t)| = |t|$
    for an arbitrary tail list $t$.
3.  **Inductive Step ($xs = \text{Cons}(h, t)$):** $$\begin{aligned}
    |\text{reverse}(\text{Cons}(h, t))| &= |\text{append}(\text{reverse}(t), \text{Cons}(h, \text{Nil}))| && \text{by (Rev 2)} \\
    &= |\text{reverse}(t)| + |\text{Cons}(h, \text{Nil})| && \text{by the append length lemma} \\
    &= |t| + 1 && \text{by IH and definition of length} \\
    &= 1 + |t| && \text{by commutativity of addition} \\
    &= |\text{Cons}(h, t)| && \text{by definition of length}
    \end{aligned}$$ $\blacksquare$ **Q.E.D.** By structural induction,
    the theorem holds for all finite lists.

**Specification:**

Implement an inductive functional list `VerifiedList<T>`:

1.  Model nodes using immutable sum types or
    `std::shared_ptr<const Node>`.
2.  Implement pure `append(VerifiedList, VerifiedList)` and
    `reverse(VerifiedList)`.
3.  Implement `size() -> std::size_t`.
4.  Construct an automated runtime inductive validator that asserts
    $|\text{reverse}(xs)| == |xs|$ and
    $|\text{append}(xs, ys)| == |xs| + |ys|$ for arbitrary generated
    lists.

``` cpp
#include <memory>
#include <cstddef>

template <typename T>
class VerifiedList {
    struct Node {
        T head;
        std::shared_ptr<const Node> tail;
    };
    std::shared_ptr<const Node> m_head;
    explicit VerifiedList(std::shared_ptr<const Node> h) : m_head(std::move(h)) {}
public:
    VerifiedList() = default;
    [[nodiscard]] VerifiedList cons(T val) const {
        return VerifiedList(std::make_shared<const Node>(std::move(val), m_head));
    }
    [[nodiscard]] std::size_t size() const noexcept {
        std::size_t count = 0;
        for (auto curr = m_head; curr != nullptr; curr = curr->tail) ++count;
        return count;
    }
    [[nodiscard]] bool empty() const noexcept { return m_head == nullptr; }
};
```

**Verification Tasks:**

- **Task 1 (Base Case Verification):** Construct an empty
  `VerifiedList<int>`. Verify that `list.size() == 0` and reversing the
  empty list yields a list of size `0`.
- **Task 2 (Inductive Step Verification):** Build lists of sizes
  $N \in [1, 500]$. Reverse each list. Assert that the returned list’s
  size matches $N$ exactly.
- **Task 3 (Generative Property Testing):** Using `PropertyTester` from
  Section 6, generate 1,000 random lists. Verify that
  `reverse(reverse(xs)) == xs` and that the length invariant holds
  unconditionally.

#### Exercise 15.2: Loop Invariants and Ranking Functions in Insertion Sort

**Context and Theory:**

To prove that an imperative loop computes the correct result and is
guaranteed to terminate, we use **Hoare Logic**.

A loop proof requires two elements:

1.  **The Loop Invariant ($I$):** A logical predicate that is true
    before the loop begins, remains true after each iteration, and,
    combined with the negation of the loop guard condition ($\neg B$),
    proves the postcondition:
    $$\{I \land B\} \; C \; \{I\} \implies \{I\} \; \text{while } (B) \ \{ C \} \; \{I \land \neg B\}$$
2.  **The Variant (Ranking) Function ($V$):** A function mapping the
    program state to a **well-founded set** (typically non-negative
    integers $\mathbb{N}_0$) that strictly decreases on every iteration:
    $$V(s_{k+1}) < V(s_k) \quad \text{and} \quad V(s) \ge 0$$ Because a
    sequence of natural numbers cannot decrease infinitely, the loop
    **must terminate**.

For Insertion Sort:

- **Loop Invariant ($I$):** At the start of iteration $i$, the subarray
  `arr[0 .. i-1]` consists of the original elements of `arr[0 .. i-1]`,
  but in strictly sorted order:
  $$\forall j, k \quad (0 \le j \le k < i) \implies \text{arr}[j] \le \text{arr}[k]$$
- **Ranking Function ($V$):** $V(i) = N - i$, where
  $N = \text{arr.size()}$. At each step, $i$ increments by 1, so $V$
  strictly decreases to 0.

**Specification:**

Implement `contracted_insertion_sort`:

1.  Use `std::span<T> arr`.
2.  Write a helper
    `is_sorted_prefix(span<T> arr, size_t prefix_len) -> bool`.
3.  Add explicit contract assertions verifying that the invariant holds
    before iteration 1 (Initialisation), after each inner sift-down
    (Maintenance), and upon loop exit (Termination).
4.  Assert that the ranking function $V(i) = N - i$ decreases on every
    iteration.

``` cpp
#include <span>
#include <algorithm>
// #include "Contracts.hpp"

template <typename T>
void contracted_insertion_sort(std::span<T> arr) {
    const std::size_t n = arr.size();
    if (n <= 1) return;

    for (std::size_t i = 1; i < n; ++i) {
        // 1. Invariant Verification: Prefix arr[0 .. i-1] is sorted
        CONTRACT_INVARIANT(std::is_sorted(arr.begin(), arr.begin() + i));
        
        T key = arr[i];
        std::size_t j = i;
        while (j > 0 && arr[j - 1] > key) {
            arr[j] = arr[j - 1];
            --j;
        }
        arr[j] = key;
    }
    // 2. Postcondition: Entire array is sorted
    CONTRACT_ENSURES(std::is_sorted(arr.begin(), arr.end()));
}
```

**Verification Tasks:**

- **Task 1 (Reverse-Sorted Stress Test):** Sort an array initialised in
  strictly descending order `[10, 9, 8, 7, 6, 5, 4, 3, 2, 1]`. Verify
  that all invariant checks pass and the final array is `[1 .. 10]`.
- **Task 2 (Invariant Failure Detection):** Intentionally modify the
  algorithm to corrupt `arr[0]` mid-sort. Verify that
  `CONTRACT_INVARIANT` catches the violation and halts execution.
- **Task 3 (Ranking Function Monotonicity):** Record the value of
  $V(i) = N - i$ at every iteration. Verify programmatically that $V(i)$
  strictly decreases by exactly $1$ on each loop pass, reaching $0$ at
  termination.

#### Exercise 15.3: Height-Balanced AVL Tree Invariant Verification

**Context and Theory:**

An **AVL Tree** (Adelson-Velsky and Landis, 1962) is a self-balancing
binary search tree. To guarantee $O(\log N)$ search, insertion, and
deletion times, the tree maintains the **Height-Balance Invariant**:

For every node $N$ in the tree, the heights of its left and right
subtrees differ by at most 1:
$$\text{BalanceFactor}(N) = \text{Height}(N.\text{left}) - \text{Height}(N.\text{right}) \in \{-1, 0, 1\}$$

When an insertion or deletion violates this invariant
($\text{BalanceFactor} \in \{-2, 2\}$), the tree performs local
rebalancing via **Tree Rotations** (Left, Right, Left-Right, or
Right-Left). See fig. 4 for a representation of Right Rotation.

<figure id="fig:tree-rotations">
<img src="data:image/svg+xml;base64,PD94bWwgdmVyc2lvbj0iMS4wIiBlbmNvZGluZz0iVVRGLTgiIHN0YW5kYWxvbmU9Im5vIj8+Cjwh
RE9DVFlQRSBzdmcgUFVCTElDICItLy9XM0MvL0RURCBTVkcgMS4xLy9FTiIKICJodHRwOi8vd3d3
LnczLm9yZy9HcmFwaGljcy9TVkcvMS4xL0RURC9zdmcxMS5kdGQiPgo8IS0tIEdlbmVyYXRlZCBi
eSBncmFwaHZpeiB2ZXJzaW9uIDIuNDMuMCAoMCkKIC0tPgo8IS0tIFRpdGxlOiBHIFBhZ2VzOiAx
IC0tPgo8c3ZnIHdpZHRoPSIxODFwdCIgaGVpZ2h0PSIxMjFwdCIKIHZpZXdCb3g9IjAuMDAgMC4w
MCAxODEuMDAgMTIxLjAwIiB4bWxucz0iaHR0cDovL3d3dy53My5vcmcvMjAwMC9zdmciIHhtbG5z
OnhsaW5rPSJodHRwOi8vd3d3LnczLm9yZy8xOTk5L3hsaW5rIj4KPGcgaWQ9ImdyYXBoMCIgY2xh
c3M9ImdyYXBoIiB0cmFuc2Zvcm09InNjYWxlKDEgMSkgcm90YXRlKDApIHRyYW5zbGF0ZSg0IDEx
NykiPgo8dGl0bGU+RzwvdGl0bGU+CjwhLS0gWiAtLT4KPGcgaWQ9Im5vZGUxIiBjbGFzcz0ibm9k
ZSI+Cjx0aXRsZT5aPC90aXRsZT4KPGVsbGlwc2UgZmlsbD0ibm9uZSIgc3Ryb2tlPSJibGFjayIg
Y3g9IjUyIiBjeT0iLTkxIiByeD0iMTEiIHJ5PSIxMSIvPgo8dGV4dCB0ZXh0LWFuY2hvcj0ibWlk
ZGxlIiB4PSI1MiIgeT0iLTg4LjUiIGZvbnQtZmFtaWx5PSJJb3NldmthIENoYXJvbiBNb25vIE1l
ZGl1bSIgZm9udC1zaXplPSIxMC4wMCI+WjwvdGV4dD4KPHRleHQgdGV4dC1hbmNob3I9Im1pZGRs
ZSIgeD0iMjAuNSIgeT0iLTEwNSIgZm9udC1mYW1pbHk9Iklvc2V2a2EgQ2hhcm9uIE1vbm8gTWVk
aXVtIiBmb250LXNpemU9IjEwLjAwIj5bQkYgPSArMl08L3RleHQ+CjwvZz4KPCEtLSBZIC0tPgo8
ZyBpZD0ibm9kZTIiIGNsYXNzPSJub2RlIj4KPHRpdGxlPlk8L3RpdGxlPgo8ZWxsaXBzZSBmaWxs
PSJub25lIiBzdHJva2U9ImJsYWNrIiBjeD0iMjIiIGN5PSItNTEiIHJ4PSIxMSIgcnk9IjExIi8+
Cjx0ZXh0IHRleHQtYW5jaG9yPSJtaWRkbGUiIHg9IjIyIiB5PSItNDguNSIgZm9udC1mYW1pbHk9
Iklvc2V2a2EgQ2hhcm9uIE1vbm8gTWVkaXVtIiBmb250LXNpemU9IjEwLjAwIj5ZPC90ZXh0Pgo8
L2c+CjwhLS0gWiYjNDU7Jmd0O1kgLS0+CjxnIGlkPSJlZGdlMSIgY2xhc3M9ImVkZ2UiPgo8dGl0
bGU+WiYjNDU7Jmd0O1k8L3RpdGxlPgo8cGF0aCBmaWxsPSJub25lIiBzdHJva2U9ImJsYWNrIiBk
PSJNNDUuNjQsLTgxLjk0QzQwLjU1LC03NS40OSAzMy40NCwtNjYuNDkgMjguMzUsLTYwLjA1Ii8+
CjwvZz4KPCEtLSBUMyAtLT4KPGcgaWQ9Im5vZGUzIiBjbGFzcz0ibm9kZSI+Cjx0aXRsZT5UMzwv
dGl0bGU+CjxlbGxpcHNlIGZpbGw9Im5vbmUiIHN0cm9rZT0iYmxhY2siIGN4PSI2MiIgY3k9Ii01
MSIgcng9IjExIiByeT0iMTEiLz4KPHRleHQgdGV4dC1hbmNob3I9Im1pZGRsZSIgeD0iNjIiIHk9
Ii00OC41IiBmb250LWZhbWlseT0iSW9zZXZrYSBDaGFyb24gTW9ubyBNZWRpdW0iIGZvbnQtc2l6
ZT0iMTAuMDAiPlQzPC90ZXh0Pgo8L2c+CjwhLS0gWiYjNDU7Jmd0O1QzIC0tPgo8ZyBpZD0iZWRn
ZTIiIGNsYXNzPSJlZGdlIj4KPHRpdGxlPlomIzQ1OyZndDtUMzwvdGl0bGU+CjxwYXRoIGZpbGw9
Im5vbmUiIHN0cm9rZT0iYmxhY2siIGQ9Ik01NC42MywtODAuMDJDNTYuMTEsLTc0LjM5IDU3Ljk1
LC02Ny4zOCA1OS40MiwtNjEuNzkiLz4KPC9nPgo8IS0tIFggLS0+CjxnIGlkPSJub2RlNCIgY2xh
c3M9Im5vZGUiPgo8dGl0bGU+WDwvdGl0bGU+CjxlbGxpcHNlIGZpbGw9Im5vbmUiIHN0cm9rZT0i
YmxhY2siIGN4PSIyMiIgY3k9Ii0xMSIgcng9IjExIiByeT0iMTEiLz4KPHRleHQgdGV4dC1hbmNo
b3I9Im1pZGRsZSIgeD0iMjIiIHk9Ii04LjUiIGZvbnQtZmFtaWx5PSJJb3NldmthIENoYXJvbiBN
b25vIE1lZGl1bSIgZm9udC1zaXplPSIxMC4wMCI+WDwvdGV4dD4KPC9nPgo8IS0tIFkmIzQ1OyZn
dDtYIC0tPgo8ZyBpZD0iZWRnZTMiIGNsYXNzPSJlZGdlIj4KPHRpdGxlPlkmIzQ1OyZndDtYPC90
aXRsZT4KPHBhdGggZmlsbD0ibm9uZSIgc3Ryb2tlPSJibGFjayIgZD0iTTIyLC0zOS42MkMyMiwt
MzQuMTcgMjIsLTI3LjUyIDIyLC0yMi4xMSIvPgo8L2c+CjwhLS0gVDIgLS0+CjxnIGlkPSJub2Rl
NSIgY2xhc3M9Im5vZGUiPgo8dGl0bGU+VDI8L3RpdGxlPgo8ZWxsaXBzZSBmaWxsPSJub25lIiBz
dHJva2U9ImJsYWNrIiBjeD0iNjIiIGN5PSItMTEiIHJ4PSIxMSIgcnk9IjExIi8+Cjx0ZXh0IHRl
eHQtYW5jaG9yPSJtaWRkbGUiIHg9IjYyIiB5PSItOC41IiBmb250LWZhbWlseT0iSW9zZXZrYSBD
aGFyb24gTW9ubyBNZWRpdW0iIGZvbnQtc2l6ZT0iMTAuMDAiPlQyPC90ZXh0Pgo8L2c+CjwhLS0g
WSYjNDU7Jmd0O1QyIC0tPgo8ZyBpZD0iZWRnZTQiIGNsYXNzPSJlZGdlIj4KPHRpdGxlPlkmIzQ1
OyZndDtUMjwvdGl0bGU+CjxwYXRoIGZpbGw9Im5vbmUiIHN0cm9rZT0iYmxhY2siIGQ9Ik0yOS43
MSwtNDIuNjdDMzYuOCwtMzUuOTQgNDcuMjQsLTI2LjAzIDU0LjMxLC0xOS4zIi8+CjwvZz4KPCEt
LSBZXzEgLS0+CjxnIGlkPSJub2RlNiIgY2xhc3M9Im5vZGUiPgo8dGl0bGU+WV8xPC90aXRsZT4K
PGVsbGlwc2UgZmlsbD0ibm9uZSIgc3Ryb2tlPSJibGFjayIgY3g9IjExMiIgY3k9Ii05MSIgcng9
IjExIiByeT0iMTEiLz4KPHRleHQgdGV4dC1hbmNob3I9Im1pZGRsZSIgeD0iMTEyIiB5PSItODgu
NSIgZm9udC1mYW1pbHk9Iklvc2V2a2EgQ2hhcm9uIE1vbm8gTWVkaXVtIiBmb250LXNpemU9IjEw
LjAwIj5ZPC90ZXh0Pgo8dGV4dCB0ZXh0LWFuY2hvcj0ibWlkZGxlIiB4PSI4My41IiB5PSItMTA1
IiBmb250LWZhbWlseT0iSW9zZXZrYSBDaGFyb24gTW9ubyBNZWRpdW0iIGZvbnQtc2l6ZT0iMTAu
MDAiPltCRiA9IDBdPC90ZXh0Pgo8L2c+CjwhLS0gWF8xIC0tPgo8ZyBpZD0ibm9kZTciIGNsYXNz
PSJub2RlIj4KPHRpdGxlPlhfMTwvdGl0bGU+CjxlbGxpcHNlIGZpbGw9Im5vbmUiIHN0cm9rZT0i
YmxhY2siIGN4PSIxMDIiIGN5PSItNTEiIHJ4PSIxMSIgcnk9IjExIi8+Cjx0ZXh0IHRleHQtYW5j
aG9yPSJtaWRkbGUiIHg9IjEwMiIgeT0iLTQ4LjUiIGZvbnQtZmFtaWx5PSJJb3NldmthIENoYXJv
biBNb25vIE1lZGl1bSIgZm9udC1zaXplPSIxMC4wMCI+WDwvdGV4dD4KPC9nPgo8IS0tIFlfMSYj
NDU7Jmd0O1hfMSAtLT4KPGcgaWQ9ImVkZ2U1IiBjbGFzcz0iZWRnZSI+Cjx0aXRsZT5ZXzEmIzQ1
OyZndDtYXzE8L3RpdGxlPgo8cGF0aCBmaWxsPSJub25lIiBzdHJva2U9ImJsYWNrIiBkPSJNMTA5
LjM3LC04MC4wMkMxMDcuODksLTc0LjM5IDEwNi4wNSwtNjcuMzggMTA0LjU4LC02MS43OSIvPgo8
L2c+CjwhLS0gWl8xIC0tPgo8ZyBpZD0ibm9kZTgiIGNsYXNzPSJub2RlIj4KPHRpdGxlPlpfMTwv
dGl0bGU+CjxlbGxpcHNlIGZpbGw9Im5vbmUiIHN0cm9rZT0iYmxhY2siIGN4PSIxNDIiIGN5PSIt
NTEiIHJ4PSIxMSIgcnk9IjExIi8+Cjx0ZXh0IHRleHQtYW5jaG9yPSJtaWRkbGUiIHg9IjE0MiIg
eT0iLTQ4LjUiIGZvbnQtZmFtaWx5PSJJb3NldmthIENoYXJvbiBNb25vIE1lZGl1bSIgZm9udC1z
aXplPSIxMC4wMCI+WjwvdGV4dD4KPC9nPgo8IS0tIFlfMSYjNDU7Jmd0O1pfMSAtLT4KPGcgaWQ9
ImVkZ2U2IiBjbGFzcz0iZWRnZSI+Cjx0aXRsZT5ZXzEmIzQ1OyZndDtaXzE8L3RpdGxlPgo8cGF0
aCBmaWxsPSJub25lIiBzdHJva2U9ImJsYWNrIiBkPSJNMTE4LjM2LC04MS45NEMxMjMuNDUsLTc1
LjQ5IDEzMC41NiwtNjYuNDkgMTM1LjY1LC02MC4wNSIvPgo8L2c+CjwhLS0gVDJfMSAtLT4KPGcg
aWQ9Im5vZGU5IiBjbGFzcz0ibm9kZSI+Cjx0aXRsZT5UMl8xPC90aXRsZT4KPGVsbGlwc2UgZmls
bD0ibm9uZSIgc3Ryb2tlPSJibGFjayIgY3g9IjEyMiIgY3k9Ii0xMSIgcng9IjExIiByeT0iMTEi
Lz4KPHRleHQgdGV4dC1hbmNob3I9Im1pZGRsZSIgeD0iMTIyIiB5PSItOC41IiBmb250LWZhbWls
eT0iSW9zZXZrYSBDaGFyb24gTW9ubyBNZWRpdW0iIGZvbnQtc2l6ZT0iMTAuMDAiPlQyPC90ZXh0
Pgo8L2c+CjwhLS0gWl8xJiM0NTsmZ3Q7VDJfMSAtLT4KPGcgaWQ9ImVkZ2U3IiBjbGFzcz0iZWRn
ZSI+Cjx0aXRsZT5aXzEmIzQ1OyZndDtUMl8xPC90aXRsZT4KPHBhdGggZmlsbD0ibm9uZSIgc3Ry
b2tlPSJibGFjayIgZD0iTTEzNy4xNiwtNDAuOEMxMzQuMDEsLTM0LjgzIDEyOS45MywtMjcuMDcg
MTI2LjgsLTIxLjExIi8+CjwvZz4KPCEtLSBUM18xIC0tPgo8ZyBpZD0ibm9kZTEwIiBjbGFzcz0i
bm9kZSI+Cjx0aXRsZT5UM18xPC90aXRsZT4KPGVsbGlwc2UgZmlsbD0ibm9uZSIgc3Ryb2tlPSJi
bGFjayIgY3g9IjE2MiIgY3k9Ii0xMSIgcng9IjExIiByeT0iMTEiLz4KPHRleHQgdGV4dC1hbmNo
b3I9Im1pZGRsZSIgeD0iMTYyIiB5PSItOC41IiBmb250LWZhbWlseT0iSW9zZXZrYSBDaGFyb24g
TW9ubyBNZWRpdW0iIGZvbnQtc2l6ZT0iMTAuMDAiPlQzPC90ZXh0Pgo8L2c+CjwhLS0gWl8xJiM0
NTsmZ3Q7VDNfMSAtLT4KPGcgaWQ9ImVkZ2U4IiBjbGFzcz0iZWRnZSI+Cjx0aXRsZT5aXzEmIzQ1
OyZndDtUM18xPC90aXRsZT4KPHBhdGggZmlsbD0ibm9uZSIgc3Ryb2tlPSJibGFjayIgZD0iTTE0
Ni44NCwtNDAuOEMxNDkuOTksLTM0LjgzIDE1NC4wNywtMjcuMDcgMTU3LjIsLTIxLjExIi8+Cjwv
Zz4KPC9nPgo8L3N2Zz4K
" alt="" />
<figcaption>Figure 4: Tree Rotation (Right Rotation about Node
Z)</figcaption>
</figure>

A fully verified AVL tree enforces two global class invariants after
every mutating operation:

1.  **The Binary Search Tree (BST) Invariant:** For every node $N$, all
    keys in $N.\text{left}$ are strictly less than $N.\text{key}$, and
    all keys in $N.\text{right}$ are strictly greater.
2.  **The AVL Balance Invariant:** For every node $N$,
    $|\text{Height}(N.\text{left}) - \text{Height}(N.\text{right})| \le 1$.

**Specification:**

Implement `VerifiedAVLTree<T>`:

1.  Maintain nodes with `T key`, `int height`, and raw/smart pointers to
    `left` and `right`.
2.  Implement recursive helper `int compute_actual_height(const Node*)`
    and `bool verify_balance_invariant(const Node*)`.
3.  Add a public method `void check_invariants() const` that executes:
    - `CONTRACT_INVARIANT(verify_bst_ordering(m_root));`
    - `CONTRACT_INVARIANT(verify_balance_invariant(m_root));`
4.  Assert this invariant at the exit of `insert(T key)` and
    `erase(const T& key)`.

``` cpp
#include <algorithm>
#include <cmath>
// #include "Contracts.hpp"

template <typename T>
class VerifiedAVLTree {
    struct Node {
        T key;
        int height{1};
        Node* left{nullptr};
        Node* right{nullptr};
        explicit Node(T k) : key(std::move(k)) {}
    };
    Node* m_root{nullptr};

    static int get_height(const Node* n) noexcept { return n ? n->height : 0; }
    
    static bool verify_balance(const Node* n) {
        if (!n) return true;
        const int bf = get_height(n->left) - get_height(n->right);
        if (std::abs(bf) > 1) return false;
        return verify_balance(n->left) && verify_balance(n->right);
    }
public:
    void check_invariants() const {
        CONTRACT_INVARIANT(verify_balance(m_root));
    }
};
```

**Verification Tasks:**

- **Task 1 (Degenerate Sequence Balancing):** Insert elements in
  strictly increasing order: $1, 2, 3, 4, 5, 6, 7, 8, 9, 10$. In an
  unbalanced BST, this creates a linked list of height 10. Verify that
  `VerifiedAVLTree` executes rotations, maintains the balance invariant,
  and terminates with $\text{Height} \le 4$.
- **Task 2 (Invariant Failure on Corrupted Rotation):** Deliberately
  disable the double-rotation branch in the rebalancing logic. Insert
  elements triggering a Left-Right imbalance. Verify that
  `CONTRACT_INVARIANT` catches the violation.
- **Task 3 (Randomised Stress Verification):** Using `PropertyTester`,
  generate 500 permutations of 200 random integers. Insert and delete
  them. Verify that the balance invariant is preserved after every
  operation.

#### Exercise 15.4: Counterexample Shrinking in Property-Based Testing

**Context and Theory:**

When a Property-Based Test fails, the generator has typically supplied a
large, complex, randomised data structure (e.g., a vector of 200
elements containing scattered negative numbers and duplicates). While
this confirms that a defect exists, diagnosing the root cause from a
massive input is difficult.

To make failures actionable, modern property-based testing frameworks
implement **Counterexample Shrinking** (introduced by Claessen and
Hughes in *QuickCheck*).

When a property fails for a candidate input $X$, the testing harness
enters a **Shrinking Loop**:

1.  It systematically generates smaller, simpler variants of $X$ (e.g.,
    halving a vector’s length, dropping prefixes/suffixes, decrementing
    integers towards zero).
2.  It re-tests each variant against the property.
3.  If a smaller variant *also* fails, it discards the larger input and
    continues shrinking the smaller one.
4.  When no further reduction fails the property, the harness reports
    the **Minimal Failing Counterexample**.

`svgbob { id="fig:shrinking-loop" caption} "Failing Input: [ 42, -7, 19, -3, 88, 102, -1, 54 ]  (Property: all elements positive)"        |        v  Drop half "   [ -7, 19, -3, 88 ]                               (Fails)"        |        v  Drop half "   [ -7, 19 ]                                       (Fails)"        |        v  Drop element "   [ -7 ]                                           (Fails)"        |        v  Shrink integer towards zero "   [ -1 ]                                           (Minimal Failing Counterexample!)"`

**Specification:**

Extend the `PropertyTester` class from Section 6 to implement shrinking:

1.  Define a type-trait or concept `Shrinkable<T>` exposing
    `std::vector<T> shrink(const T& val)`.
2.  For `std::vector<int>`, shrinking strategies must include:
    - Removing chunks of elements (halves, quarters, individual
      elements).
    - Shrinking individual failing integers towards zero ($x \to x / 2$,
      $x \to x - 1$).
3.  When a test fails, recursively shrink the input until no smaller
    candidate fails.
4.  Output the initial failing candidate, the total number of shrink
    steps, and the minimal counterexample.

``` cpp
#include <vector>
#include <iostream>

template <typename T>
struct Shrinker;

template <>
struct Shrinker<std::vector<int>> {
    static std::vector<std::vector<int>> candidates(const std::vector<int>& vec) {
        std::vector<std::vector<int>> results;
        if (vec.empty()) return results;
        // 1. Candidate: empty vector
        results.push_back({});
        // 2. Halves
        results.push_back(std::vector<int>(vec.begin(), vec.begin() + vec.size() / 2));
        results.push_back(std::vector<int>(vec.begin() + vec.size() / 2, vec.end()));
        // 3. Drop single elements
        for (std::size_t i = 0; i < vec.size(); ++i) {
            auto copy = vec;
            copy.erase(copy.begin() + i);
            results.push_back(std::move(copy));
        }
        return results;
    }
};
```

**Verification Tasks:**

- **Task 1 (Minimal Vector Isolation):** Define a property: *“Vectors
  must not contain any negative values”*. Pass a randomised vector of 50
  positive integers containing exactly one negative integer `-8`. Verify
  that the shrinker reduces the failure to a 1-element vector `[-1]` or
  `[-8]`.
- **Task 2 (Threshold Boundary Shrinking):** Define a property:
  *“Integers must be less than 100”*. Supply a failing input of `4528`.
  Verify that the shrinker decrements the value to the exact boundary
  `100`.
- **Task 3 (Shrink Step Count):** Confirm that shrinking terminates in a
  finite number of steps ($< 50$ iterations) and does not enter an
  infinite loop.

#### Exercise 15.5: Non-Type Template Parameter Contracts: Dimension-Safe Fixed Matrix

**Context and Theory:**

In systems engineering, runtime assertions (`assert`, exceptions) carry
two drawbacks:

1.  They incur runtime execution overhead unless compiled out.
2.  They fail late: the error is detected only when the invalid code
    path executes during deployment.

Where possible, contracts should be shifted to **Compile-Time Static
Proofs** using C++20 Non-Type Template Parameters (NTTP) and
`static_assert`.

Consider matrix multiplication:

$$A \in \mathbb{R}^{R_1 \times C_1}, \quad B \in \mathbb{R}^{R_2 \times C_2}$$

The matrix product $A \times B$ is mathematically undefined unless the
column count of the first matrix equals the row count of the second
($C_1 == R_2$).

By embedding dimensions into the type signature, we turn dimensional
correctness into an **unconditional compile-time contract**. It is
physically impossible to compile an invalid multiplication (see fig. 5).

<figure id="fig:static-multiply">
<img src="data:image/svg+xml;base64,PHN2ZyB4bWxucz0iaHR0cDovL3d3dy53My5vcmcvMjAwMC9zdmciIHdpZHRoPSI1MzcuNzY4IiBo
ZWlnaHQ9Ijg3LjgwOCIgY2xhc3M9InN2Z2JvYiIgdmlld0JveD0iMCAtNCA1MzcuNzY4IDg3Ljgw
OCI+CiAgPHN0eWxlPi5zdmdib2IgbGluZSwgLnN2Z2JvYiBwYXRoLCAuc3ZnYm9iIGNpcmNsZSwg
LnN2Z2JvYiByZWN0LCAuc3ZnYm9iIHBvbHlnb24gewogIHN0cm9rZTogYmxhY2s7CiAgc3Ryb2tl
LXdpZHRoOiAxLjU7CiAgc3Ryb2tlLW9wYWNpdHk6IDE7CiAgZmlsbC1vcGFjaXR5OiAxOwogIHN0
cm9rZS1saW5lY2FwOiByb3VuZDsKICBzdHJva2UtbGluZWpvaW46IG1pdGVyOwp9Cgouc3ZnYm9i
IHRleHQgewogIHdoaXRlLXNwYWNlOiBwcmU7CiAgZmlsbDogYmxhY2s7CiAgZm9udC1mYW1pbHk6
IElvc2V2a2EgQ2hhcm9uIE1vbm8gTWVkaXVtLCBtb25vc3BhY2U7CiAgZm9udC1zaXplOiAxNHB4
Owp9Cgouc3ZnYm9iIHJlY3QuYmFja2Ryb3AgewogIHN0cm9rZTogbm9uZTsKICBmaWxsOiB0cmFu
c3BhcmVudDsKfQoKLnN2Z2JvYiAuYnJva2VuIHsKICBzdHJva2UtZGFzaGFycmF5OiA4Owp9Cgou
c3ZnYm9iIC5maWxsZWQgewogIGZpbGw6IGJsYWNrOwp9Cgouc3ZnYm9iIC5iZ19maWxsZWQgewog
IGZpbGw6IHRyYW5zcGFyZW50OwogIHN0cm9rZS13aWR0aDogMTsKfQoKLnN2Z2JvYiAubm9maWxs
IHsKICBmaWxsOiB0cmFuc3BhcmVudDsKfQoKLnN2Z2JvYiAuZW5kX21hcmtlZF9hcnJvdyB7CiAg
bWFya2VyLWVuZDogdXJsKCNhcnJvdyk7Cn0KCi5zdmdib2IgLnN0YXJ0X21hcmtlZF9hcnJvdyB7
CiAgbWFya2VyLXN0YXJ0OiB1cmwoI2Fycm93KTsKfQoKLnN2Z2JvYiAuZW5kX21hcmtlZF9kaWFt
b25kIHsKICBtYXJrZXItZW5kOiB1cmwoI2RpYW1vbmQpOwp9Cgouc3ZnYm9iIC5zdGFydF9tYXJr
ZWRfZGlhbW9uZCB7CiAgbWFya2VyLXN0YXJ0OiB1cmwoI2RpYW1vbmQpOwp9Cgouc3ZnYm9iIC5l
bmRfbWFya2VkX2NpcmNsZSB7CiAgbWFya2VyLWVuZDogdXJsKCNjaXJjbGUpOwp9Cgouc3ZnYm9i
IC5zdGFydF9tYXJrZWRfY2lyY2xlIHsKICBtYXJrZXItc3RhcnQ6IHVybCgjY2lyY2xlKTsKfQoK
LnN2Z2JvYiAuZW5kX21hcmtlZF9vcGVuX2NpcmNsZSB7CiAgbWFya2VyLWVuZDogdXJsKCNvcGVu
X2NpcmNsZSk7Cn0KCi5zdmdib2IgLnN0YXJ0X21hcmtlZF9vcGVuX2NpcmNsZSB7CiAgbWFya2Vy
LXN0YXJ0OiB1cmwoI29wZW5fY2lyY2xlKTsKfQoKLnN2Z2JvYiAuZW5kX21hcmtlZF9iaWdfb3Bl
bl9jaXJjbGUgewogIG1hcmtlci1lbmQ6IHVybCgjYmlnX29wZW5fY2lyY2xlKTsKfQoKLnN2Z2Jv
YiAuc3RhcnRfbWFya2VkX2JpZ19vcGVuX2NpcmNsZSB7CiAgbWFya2VyLXN0YXJ0OiB1cmwoI2Jp
Z19vcGVuX2NpcmNsZSk7Cn0KCjwvc3R5bGU+CiAgPGRlZnM+CiAgICA8bWFya2VyIGlkPSJhcnJv
dyIgdmlld0JveD0iLTIgLTIgOCA4IiByZWZYPSI0IiByZWZZPSIyIiBtYXJrZXJXaWR0aD0iNyIg
bWFya2VySGVpZ2h0PSI3IiBvcmllbnQ9ImF1dG8tc3RhcnQtcmV2ZXJzZSI+CiAgICAgIDxwb2x5
Z29uIHBvaW50cz0iMCwwIDAsNCA0LDIgMCwwIj48L3BvbHlnb24+CiAgICA8L21hcmtlcj4KICAg
IDxtYXJrZXIgaWQ9ImRpYW1vbmQiIHZpZXdCb3g9Ii0yIC0yIDggOCIgcmVmWD0iNCIgcmVmWT0i
MiIgbWFya2VyV2lkdGg9IjciIG1hcmtlckhlaWdodD0iNyIgb3JpZW50PSJhdXRvLXN0YXJ0LXJl
dmVyc2UiPgogICAgICA8cG9seWdvbiBwb2ludHM9IjAsMiAyLDAgNCwyIDIsNCAwLDIiPjwvcG9s
eWdvbj4KICAgIDwvbWFya2VyPgogICAgPG1hcmtlciBpZD0iY2lyY2xlIiB2aWV3Qm94PSIwIDAg
OCA4IiByZWZYPSI0IiByZWZZPSI0IiBtYXJrZXJXaWR0aD0iNyIgbWFya2VySGVpZ2h0PSI3IiBv
cmllbnQ9ImF1dG8tc3RhcnQtcmV2ZXJzZSI+CiAgICAgIDxjaXJjbGUgY3g9IjQiIGN5PSI0IiBy
PSIyIiBjbGFzcz0iZmlsbGVkIj48L2NpcmNsZT4KICAgIDwvbWFya2VyPgogICAgPG1hcmtlciBp
ZD0ib3Blbl9jaXJjbGUiIHZpZXdCb3g9IjAgMCA4IDgiIHJlZlg9IjQiIHJlZlk9IjQiIG1hcmtl
cldpZHRoPSI3IiBtYXJrZXJIZWlnaHQ9IjciIG9yaWVudD0iYXV0by1zdGFydC1yZXZlcnNlIj4K
ICAgICAgPGNpcmNsZSBjeD0iNCIgY3k9IjQiIHI9IjIiIGNsYXNzPSJiZ19maWxsZWQiPjwvY2ly
Y2xlPgogICAgPC9tYXJrZXI+CiAgICA8bWFya2VyIGlkPSJiaWdfb3Blbl9jaXJjbGUiIHZpZXdC
b3g9IjAgMCA4IDgiIHJlZlg9IjQiIHJlZlk9IjQiIG1hcmtlcldpZHRoPSI3IiBtYXJrZXJIZWln
aHQ9IjciIG9yaWVudD0iYXV0by1zdGFydC1yZXZlcnNlIj4KICAgICAgPGNpcmNsZSBjeD0iNCIg
Y3k9IjQiIHI9IjMiIGNsYXNzPSJiZ19maWxsZWQiPjwvY2lyY2xlPgogICAgPC9tYXJrZXI+CiAg
PC9kZWZzPgogIDxyZWN0IGNsYXNzPSJiYWNrZHJvcCIgeD0iMCIgeT0iLTQiIHdpZHRoPSI1Mzcu
NzY4IiBoZWlnaHQ9Ijg3LjgwOCI+PC9yZWN0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUi
IHg9IjEuNzQ2IiB5PSIxMC40NzYiID5WYWxpZDwvdGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9InBy
ZXNlcnZlIiB4PSI0My42NSIgeT0iMTAuNDc2IiA+TXVsdGlwbGljYXRpb246PC90ZXh0PgogIDx0
ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjEuNzQ2IiB5PSIyNC40NDQiID5NYXRyaXgmbHQ7
Miw8L3RleHQ+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iNzEuNTg2IiB5PSIyNC40
NDQiID4zJmd0OzwvdGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSI5Mi41Mzgi
IHk9IjI0LjQ0NCIgPio8L3RleHQ+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iMTA2
LjUwNjAwNCIgeT0iMjQuNDQ0IiA+TWF0cml4Jmx0OzMsPC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFj
ZT0icHJlc2VydmUiIHg9IjE3Ni4zNDYwMSIgeT0iMjQuNDQ0IiA+NCZndDs8L3RleHQ+CiAgPGxp
bmUgeDE9IjE5NS41NTIiIHkxPSIyMC45NTIiIHgyPSIyMTYuNTA0MDEiIHkyPSIyMC45NTIiIGNs
YXNzPSJzb2xpZCI+PC9saW5lPgogIDxwb2x5Z29uIHBvaW50cz0iMjE2LjUwNDAxLDE3LjQ2MDAw
MSAyMjMuNDg4LDIwLjk1MiAyMTYuNTA0MDEsMjQuNDQ0IiBjbGFzcz0iZmlsbGVkIj48L3BvbHln
b24+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iMjMyLjIxOCIgeT0iMjQuNDQ0IiA+
UmVzdWx0OjwvdGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSIyODguMDkiIHk9
IjI0LjQ0NCIgPk1hdHJpeCZsdDsyLDwvdGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZl
IiB4PSIzNTcuOTMwMDIiIHk9IjI0LjQ0NCIgPjQmZ3Q7PC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFj
ZT0icHJlc2VydmUiIHg9IjM4NS44NjYiIHk9IjI0LjQ0NCIgPltDb21waWxlczwvdGV4dD4KICA8
dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSI0NTUuNzA2MDIiIHk9IjI0LjQ0NCIgPkNsZWFu
bHldPC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjEuNzQ2IiB5PSI1Mi4z
OCIgPkludmFsaWQ8L3RleHQ+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iNTcuNjE4
IiB5PSI1Mi4zOCIgPk11bHRpcGxpY2F0aW9uOjwvdGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9InBy
ZXNlcnZlIiB4PSIxLjc0NiIgeT0iNjYuMzQ4IiA+TWF0cml4Jmx0OzIsPC90ZXh0PgogIDx0ZXh0
IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjcxLjU4NiIgeT0iNjYuMzQ4IiA+MyZndDs8L3RleHQ+
CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iOTIuNTM4IiB5PSI2Ni4zNDgiID4qPC90
ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjEwNi41MDYwMDQiIHk9IjY2LjM0
OCIgPk1hdHJpeCZsdDsyLDwvdGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSIx
NzYuMzQ2MDEiIHk9IjY2LjM0OCIgPjMmZ3Q7PC90ZXh0PgogIDxsaW5lIHgxPSIxOTUuNTUyIiB5
MT0iNjIuODU2MDAzIiB4Mj0iMjE2LjUwNDAxIiB5Mj0iNjIuODU2MDAzIiBjbGFzcz0ic29saWQi
PjwvbGluZT4KICA8cG9seWdvbiBwb2ludHM9IjIxNi41MDQwMSw1OS4zNjQwMDIgMjIzLjQ4OCw2
Mi44NTYwMDMgMjE2LjUwNDAxLDY2LjM0OCIgY2xhc3M9ImZpbGxlZCI+PC9wb2x5Z29uPgogIDx0
ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjM4NS44NjYiIHk9IjY2LjM0OCIgPltDb21waWxh
dGlvbjwvdGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSI0NzYuNjU4MDIiIHk9
IjY2LjM0OCIgPkFib3J0ZWRdPC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9
IjIzMi4yMTgiIHk9IjY2LjM0OCIgPnN0YXRpY19hc3NlcnQgZmFpbHMhPC90ZXh0Pgo8L3N2Zz4K
" alt="" />
<figcaption>Figure 5: Compile-time Checking of Matrix
Dimensions</figcaption>
</figure>

**Specification:**

Implement `ContractedMatrix<Rows, Cols, T>`:

1.  Store elements in a flat array `std::array<T, Rows * Cols>`.

2.  Overload `operator+` requiring both operands to have identical
    `Rows` and `Cols` at compile time.

3.  Overload `operator*`:

    ``` cpp
    template <std::size_t R1, std::size_t Common, std::size_t C2, typename T>
    auto operator*(const ContractedMatrix<R1, Common, T>& a, 
                   const ContractedMatrix<Common, C2, T>& b) 
        -> ContractedMatrix<R1, C2, T>;
    ```

4.  Prevent dimension-mismatched additions or multiplications using
    `static_assert` or constrained `requires` clauses that provide clear
    compiler error diagnostics.

``` cpp
#include <array>
#include <cstddef>

template <std::size_t R, std::size_t C, typename T = double>
class ContractedMatrix {
public:
    std::array<T, R * C> data{};

    template <std::size_t OtherC>
    [[nodiscard]] constexpr auto operator*(const ContractedMatrix<C, OtherC, T>& rhs) const noexcept {
        ContractedMatrix<R, OtherC, T> result{};
        for (std::size_t i = 0; i < R; ++i) {
            for (std::size_t j = 0; j < OtherC; ++j) {
                T sum = T{0};
                for (std::size_t k = 0; k < C; ++k) sum += data[i * C + k] * rhs.data[k * OtherC + j];
                result.data[i * OtherC + j] = sum;
            }
        }
        return result;
    }
};
```

**Verification Tasks:**

- **Task 1 (Dimension Transformation Proof):** Declare
  $A \in \mathbb{R}^{3 \times 2}$ and $B \in \mathbb{R}^{2 \times 4}$.
  Compute $C = A \times B$. Verify via `static_assert` that
  `sizeof(C) == sizeof(double) * 3 * 4`.
- **Task 2 (Compile-Time Failure Verification):** Write a test case
  attempting to multiply `ContractedMatrix<3, 2>` by
  `ContractedMatrix<3, 2>`. Confirm that the compiler halts compilation
  with an explicit diagnostic.
- **Task 3 (Constexpr Compile-Time Evaluation):** Initialise two
  matrices with constant values. Multiply them inside a `constexpr`
  context. Verify via `static_assert` that the computed elements match
  mathematical expectation at compile time.

#### Exercise 15.6: Circular Ring Buffer Invariant and Postcondition Verification

**Context and Theory:**

A **Circular Ring Buffer** uses a fixed-size contiguous memory array to
implement a FIFO queue without dynamic allocations. It maintains two
indices: `m_head` (where elements are read) and `m_tail` (where elements
are written), alongside an element counter `m_count`.

Index arithmetic wraps around modulo the capacity:

$$\text{NextIndex}(i) = (i + 1) \pmod{\text{Capacity}}$$

A formal contract for a Ring Buffer specifies:

- **Precondition for Enqueue:** The buffer must not be full
  ($\text{count} < \text{Capacity}$).
- **Precondition for Dequeue:** The buffer must not be empty
  ($\text{count} > 0$).
- **Class Invariant ($I$):**
  1.  The stored element count never exceeds capacity:
      $0 \le \text{count} \le \text{Capacity}$.
  2.  The relationship between head, tail, and count is mathematically
      preserved:
      $$\text{m\_tail} \equiv (\text{m\_head} + \text{m\_count}) \pmod{\text{Capacity}}$$
- **FIFO Postcondition:** The sequence of elements returned by
  `dequeue()` must match the exact temporal sequence passed to
  `enqueue()`.

**Specification:**

Implement `ContractedRingBuffer<T, Capacity>`:

1.  Allocate inline storage using `std::array<T, Capacity>`.
2.  Implement `enqueue(T val)` and `dequeue() -> T`.
3.  Implement `check_invariants() const` verifying index consistency:
    `CONTRACT_INVARIANT(m_tail == (m_head + m_count) % Capacity);`
4.  Assert preconditions and postconditions explicitly at function entry
    and exit.

``` cpp
#include <array>
#include <cstddef>
// #include "Contracts.hpp"

template <typename T, std::size_t Capacity>
class ContractedRingBuffer {
    static_assert(Capacity > 0, "Capacity must be non-zero");
    std::array<T, Capacity> m_data{};
    std::size_t m_head{0};
    std::size_t m_tail{0};
    std::size_t m_count{0};
public:
    void check_invariants() const {
        CONTRACT_INVARIANT(m_count <= Capacity);
        CONTRACT_INVARIANT(m_tail == (m_head + m_count) % Capacity);
    }
    void enqueue(T val) {
        check_invariants();
        CONTRACT_EXPECTS(m_count < Capacity);
        m_data[m_tail] = std::move(val);
        m_tail = (m_tail + 1) % Capacity;
        ++m_count;
        check_invariants();
    }
};
```

**Verification Tasks:**

- **Task 1 (Wrap-Around Index Invariant):** Instantiate a buffer with
  `Capacity = 4`. Enqueue 3 items, dequeue 2 items, then enqueue 3 more
  items (forcing `m_tail` to wrap past the array boundary). Verify that
  `check_invariants()` passes at every step.
- **Task 2 (Precondition Violation Detection):** Enqueue items until the
  buffer is full. Attempt an additional `enqueue()`. Verify that a
  `ContractViolationException` is thrown and buffer state is
  uncorrupted.
- **Task 3 (FIFO Sequence Postcondition):** Enqueue 1,000 integers in
  sequence, interleaved with random dequeues. Verify that the retrieved
  stream matches the inserted stream without reordering.

#### Exercise 15.7: Abstract Algebra Monoid Law Verification via Generative Testing

**Context and Theory:**

In abstract algebra and category theory, a **Monoid** is an algebraic
structure consisting of a set $S$ and a binary operation
$\oplus : S \times S \to S$ that satisfies two axioms:

1.  **Associativity:** For all $a, b, c \in S$:
    $$(a \oplus b) \oplus c = a \oplus (b \oplus c)$$
2.  **Identity Element:** There exists a unique element $e \in S$ such
    that for all $a \in S$:
    $$a \oplus e = a \quad \text{and} \quad e \oplus a = a$$

Monoids are foundational in parallel and concurrent computing: any
reduction or aggregation (e.g., Map-Reduce, parallel prefix sums) can be
partitioned across worker threads **if and only if the underlying
operation forms a valid Monoid**. If an operation is non-associative,
parallel evaluations yield non-deterministic results depending on thread
scheduling.

- Valid Monoids: $(\mathbb{Z}, +, 0)$, $(\mathbb{Z}, \times, 1)$,
  $(\text{String}, +, \text{""})$, $(\text{Bool}, \lor, \text{false})$.
- Invalid Monoids: Floating-point addition $(\mathbb{R}, +, 0.0)$ is
  **not** strictly associative due to rounding:
  $$(10^{30} + -10^{30}) + 1.0 = 1.0 \quad \ne \quad 10^{30} + (-10^{30} + 1.0) = 0.0$$

**Specification:**

Design an algebraic monoid verification harness:

1.  Define a C++20 Concept `MonoidStructure<T, Op, Identity>` verifying
    operation signatures.
2.  Implement `verify_monoid_laws(Generator<T> gen, Op op, T identity)`:
    - Generates triplets $(a, b, c)$ to test associativity:
      `op(op(a, b), c) == op(a, op(b, c))`.
    - Generates singletons $a$ to test left and right identity:
      `op(a, e) == a && op(e, a) == a`.
3.  Use the harness to verify that string concatenation and integer
    addition are valid monoids.
4.  Use the harness to prove that floating-point addition fails the
    associativity law for extreme values.

``` cpp
#include <concepts>
#include <string>
#include <format>
// #include "PropertyTester.hpp"

template <typename T, typename Op>
bool test_associativity(const T& a, const T& b, const T& c, Op op) {
    return op(op(a, b), c) == op(a, op(b, c));
}

template <typename T, typename Op>
bool test_identity(const T& a, const T& identity, Op op) {
    return (op(a, identity) == a) && (op(identity, a) == a);
}
```

**Verification Tasks:**

- **Task 1 (String Monoid Proof):** Test string concatenation with the
  empty string `""` across 1,000 random string triplets. Verify that
  associativity and identity hold for all cases.
- **Task 2 (Modular Integer Monoid Proof):** Define a modular arithmetic
  structure $(\mathbb{Z}_{17}, \times, 1)$. Verify that it satisfies
  both monoid laws across all inputs.
- **Task 3 (Floating-Point Counterexample Discovery):** Run the monoid
  tester on `double` addition using generated values spanning wide
  dynamic ranges ($10^{-30}$ to $10^{30}$). Verify that the harness
  discovers and logs an associativity counterexample.

#### Exercise 15.8: Min-Heap Structural and Ordering Invariant Verifier

**Context and Theory:**

A **Binary Min-Heap** is an array-backed data structure that implements
an efficient priority queue. It satisfies two fundamental invariants:

1.  **Structural Invariant (Complete Binary Tree):** All levels of the
    tree are fully filled, except possibly the bottom level, which is
    populated from left to right. This allows the tree to be mapped onto
    a flat array without pointers:
    $$\text{Parent}(i) = \left\lfloor\frac{i - 1}{2}\right\rfloor, \quad \text{Left}(i) = 2i + 1, \quad \text{Right}(i) = 2i + 2$$
2.  **Heap-Order Invariant:** For every node $i$ other than the root:
    $$\text{Array}[\text{Parent}(i)] \le \text{Array}[i]$$ The key at
    any node is greater than or equal to the key of its parent, meaning
    the minimum element always resides at index $0$.

<figure id="fig:binary-min-heap">
<img src="data:image/svg+xml;base64,PD94bWwgdmVyc2lvbj0iMS4wIiBlbmNvZGluZz0iVVRGLTgiIHN0YW5kYWxvbmU9Im5vIj8+Cjwh
RE9DVFlQRSBzdmcgUFVCTElDICItLy9XM0MvL0RURCBTVkcgMS4xLy9FTiIKICJodHRwOi8vd3d3
LnczLm9yZy9HcmFwaGljcy9TVkcvMS4xL0RURC9zdmcxMS5kdGQiPgo8IS0tIEdlbmVyYXRlZCBi
eSBncmFwaHZpeiB2ZXJzaW9uIDIuNDMuMCAoMCkKIC0tPgo8IS0tIFRpdGxlOiBHIFBhZ2VzOiAx
IC0tPgo8c3ZnIHdpZHRoPSIxNjZwdCIgaGVpZ2h0PSIxMTJwdCIKIHZpZXdCb3g9IjAuMDAgMC4w
MCAxNjYuMDAgMTEyLjAwIiB4bWxucz0iaHR0cDovL3d3dy53My5vcmcvMjAwMC9zdmciIHhtbG5z
OnhsaW5rPSJodHRwOi8vd3d3LnczLm9yZy8xOTk5L3hsaW5rIj4KPGcgaWQ9ImdyYXBoMCIgY2xh
c3M9ImdyYXBoIiB0cmFuc2Zvcm09InNjYWxlKDEgMSkgcm90YXRlKDApIHRyYW5zbGF0ZSg0IDEw
OCkiPgo8dGl0bGU+RzwvdGl0bGU+CjwhLS0gMiAtLT4KPGcgaWQ9Im5vZGUxIiBjbGFzcz0ibm9k
ZSI+Cjx0aXRsZT4yPC90aXRsZT4KPGVsbGlwc2UgZmlsbD0ibm9uZSIgc3Ryb2tlPSJibGFjayIg
Y3g9Ijk5IiBjeT0iLTgyIiByeD0iMTEiIHJ5PSIxMSIvPgo8dGV4dCB0ZXh0LWFuY2hvcj0ibWlk
ZGxlIiB4PSI5OSIgeT0iLTc5LjUiIGZvbnQtZmFtaWx5PSJJb3NldmthIENoYXJvbiBNb25vIE1l
ZGl1bSIgZm9udC1zaXplPSIxMC4wMCI+MjwvdGV4dD4KPHRleHQgdGV4dC1hbmNob3I9Im1pZGRs
ZSIgeD0iNzQiIHk9Ii05NiIgZm9udC1mYW1pbHk9Iklvc2V2a2EgQ2hhcm9uIE1vbm8gTWVkaXVt
IiBmb250LXNpemU9IjEwLjAwIj4oaWR4IDApPC90ZXh0Pgo8L2c+CjwhLS0gNSAtLT4KPGcgaWQ9
Im5vZGUyIiBjbGFzcz0ibm9kZSI+Cjx0aXRsZT41PC90aXRsZT4KPGVsbGlwc2UgZmlsbD0ibm9u
ZSIgc3Ryb2tlPSJibGFjayIgY3g9Ijc5IiBjeT0iLTUyIiByeD0iMTEiIHJ5PSIxMSIvPgo8dGV4
dCB0ZXh0LWFuY2hvcj0ibWlkZGxlIiB4PSI3OSIgeT0iLTQ5LjUiIGZvbnQtZmFtaWx5PSJJb3Nl
dmthIENoYXJvbiBNb25vIE1lZGl1bSIgZm9udC1zaXplPSIxMC4wMCI+NTwvdGV4dD4KPHRleHQg
dGV4dC1hbmNob3I9Im1pZGRsZSIgeD0iNTQiIHk9Ii02NiIgZm9udC1mYW1pbHk9Iklvc2V2a2Eg
Q2hhcm9uIE1vbm8gTWVkaXVtIiBmb250LXNpemU9IjEwLjAwIj4oaWR4IDEpPC90ZXh0Pgo8L2c+
CjwhLS0gMiYjNDU7Jmd0OzUgLS0+CjxnIGlkPSJlZGdlMSIgY2xhc3M9ImVkZ2UiPgo8dGl0bGU+
MiYjNDU7Jmd0OzU8L3RpdGxlPgo8cGF0aCBmaWxsPSJub25lIiBzdHJva2U9ImJsYWNrIiBkPSJN
OTIuODksLTcyLjQ1QzkwLjQxLC02OC45NyA4Ny41NSwtNjQuOTcgODUuMDcsLTYxLjUiLz4KPC9n
Pgo8IS0tIDggLS0+CjxnIGlkPSJub2RlMyIgY2xhc3M9Im5vZGUiPgo8dGl0bGU+ODwvdGl0bGU+
CjxlbGxpcHNlIGZpbGw9Im5vbmUiIHN0cm9rZT0iYmxhY2siIGN4PSIxMTkiIGN5PSItNTIiIHJ4
PSIxMSIgcnk9IjExIi8+Cjx0ZXh0IHRleHQtYW5jaG9yPSJtaWRkbGUiIHg9IjExOSIgeT0iLTQ5
LjUiIGZvbnQtZmFtaWx5PSJJb3NldmthIENoYXJvbiBNb25vIE1lZGl1bSIgZm9udC1zaXplPSIx
MC4wMCI+ODwvdGV4dD4KPHRleHQgdGV4dC1hbmNob3I9Im1pZGRsZSIgeD0iMTQ0IiB5PSItNjYi
IGZvbnQtZmFtaWx5PSJJb3NldmthIENoYXJvbiBNb25vIE1lZGl1bSIgZm9udC1zaXplPSIxMC4w
MCI+KGlkeCAyKTwvdGV4dD4KPC9nPgo8IS0tIDImIzQ1OyZndDs4IC0tPgo8ZyBpZD0iZWRnZTIi
IGNsYXNzPSJlZGdlIj4KPHRpdGxlPjImIzQ1OyZndDs4PC90aXRsZT4KPHBhdGggZmlsbD0ibm9u
ZSIgc3Ryb2tlPSJibGFjayIgZD0iTTEwNS4xMSwtNzIuNDVDMTA3LjU5LC02OC45NyAxMTAuNDUs
LTY0Ljk3IDExMi45MywtNjEuNSIvPgo8L2c+CjwhLS0gMTIgLS0+CjxnIGlkPSJub2RlNCIgY2xh
c3M9Im5vZGUiPgo8dGl0bGU+MTI8L3RpdGxlPgo8ZWxsaXBzZSBmaWxsPSJub25lIiBzdHJva2U9
ImJsYWNrIiBjeD0iMzkiIGN5PSItMjIiIHJ4PSIxMSIgcnk9IjExIi8+Cjx0ZXh0IHRleHQtYW5j
aG9yPSJtaWRkbGUiIHg9IjM5IiB5PSItMTkuNSIgZm9udC1mYW1pbHk9Iklvc2V2a2EgQ2hhcm9u
IE1vbm8gTWVkaXVtIiBmb250LXNpemU9IjEwLjAwIj4xMjwvdGV4dD4KPHRleHQgdGV4dC1hbmNo
b3I9Im1pZGRsZSIgeD0iMTQiIHk9Ii0zNiIgZm9udC1mYW1pbHk9Iklvc2V2a2EgQ2hhcm9uIE1v
bm8gTWVkaXVtIiBmb250LXNpemU9IjEwLjAwIj4oaWR4IDMpPC90ZXh0Pgo8L2c+CjwhLS0gNSYj
NDU7Jmd0OzEyIC0tPgo8ZyBpZD0iZWRnZTMiIGNsYXNzPSJlZGdlIj4KPHRpdGxlPjUmIzQ1OyZn
dDsxMjwvdGl0bGU+CjxwYXRoIGZpbGw9Im5vbmUiIHN0cm9rZT0iYmxhY2siIGQ9Ik03MC4xMiwt
NDQuNzlDNjMuNCwtNDAuMDggNTQuMjIsLTMzLjY2IDQ3LjU4LC0yOSIvPgo8L2c+CjwhLS0gNyAt
LT4KPGcgaWQ9Im5vZGU1IiBjbGFzcz0ibm9kZSI+Cjx0aXRsZT43PC90aXRsZT4KPGVsbGlwc2Ug
ZmlsbD0ibm9uZSIgc3Ryb2tlPSJibGFjayIgY3g9Ijc5IiBjeT0iLTIyIiByeD0iMTEiIHJ5PSIx
MSIvPgo8dGV4dCB0ZXh0LWFuY2hvcj0ibWlkZGxlIiB4PSI3OSIgeT0iLTE5LjUiIGZvbnQtZmFt
aWx5PSJJb3NldmthIENoYXJvbiBNb25vIE1lZGl1bSIgZm9udC1zaXplPSIxMC4wMCI+NzwvdGV4
dD4KPC9nPgo8IS0tIDUmIzQ1OyZndDs3IC0tPgo8ZyBpZD0iZWRnZTQiIGNsYXNzPSJlZGdlIj4K
PHRpdGxlPjUmIzQ1OyZndDs3PC90aXRsZT4KPHBhdGggZmlsbD0ibm9uZSIgc3Ryb2tlPSJibGFj
ayIgZD0iTTc5LC00MC44OUM3OSwtMzguMzkgNzksLTM1LjcgNzksLTMzLjE5Ii8+CjwvZz4KPCEt
LSAxMCAtLT4KPGcgaWQ9Im5vZGU2IiBjbGFzcz0ibm9kZSI+Cjx0aXRsZT4xMDwvdGl0bGU+Cjxl
bGxpcHNlIGZpbGw9Im5vbmUiIHN0cm9rZT0iYmxhY2siIGN4PSIxMTkiIGN5PSItMjIiIHJ4PSIx
MSIgcnk9IjExIi8+Cjx0ZXh0IHRleHQtYW5jaG9yPSJtaWRkbGUiIHg9IjExOSIgeT0iLTE5LjUi
IGZvbnQtZmFtaWx5PSJJb3NldmthIENoYXJvbiBNb25vIE1lZGl1bSIgZm9udC1zaXplPSIxMC4w
MCI+MTA8L3RleHQ+Cjx0ZXh0IHRleHQtYW5jaG9yPSJtaWRkbGUiIHg9Ijk0IiB5PSItMyIgZm9u
dC1mYW1pbHk9Iklvc2V2a2EgQ2hhcm9uIE1vbm8gTWVkaXVtIiBmb250LXNpemU9IjEwLjAwIj4o
aWR4IDUpPC90ZXh0Pgo8L2c+CjwhLS0gOCYjNDU7Jmd0OzEwIC0tPgo8ZyBpZD0iZWRnZTUiIGNs
YXNzPSJlZGdlIj4KPHRpdGxlPjgmIzQ1OyZndDsxMDwvdGl0bGU+CjxwYXRoIGZpbGw9Im5vbmUi
IHN0cm9rZT0iYmxhY2siIGQ9Ik0xMTksLTQwLjg5QzExOSwtMzguMzkgMTE5LC0zNS43IDExOSwt
MzMuMTkiLz4KPC9nPgo8L2c+Cjwvc3ZnPgo=
" alt="" />
<figcaption>Figure 6: Array: [ 2, 5, 8, 12, 7, 10 ]</figcaption>
</figure>

During `push()`, an element is appended at the end and sifted up. During
`pop()`, the root is replaced with the last element and sifted down.

**Specification:**

Implement `ContractedMinHeap<T>`:

1.  Maintain internal storage `std::vector<T> m_tree;`.

2.  Implement `check_heap_invariant() const`:

    ``` cpp
    for (std::size_t i = 1; i < m_tree.size(); ++i) {
        CONTRACT_INVARIANT(m_tree[(i - 1) / 2] <= m_tree[i]);
    }
    ```

3.  Implement `push(T val)` and `pop() -> T`.

4.  Assert the heap-order invariant at the entry and exit of both
    mutating operations.

``` cpp
#include <vector>
#include <cstddef>
// #include "Contracts.hpp"

template <typename T>
class ContractedMinHeap {
    std::vector<T> m_tree;
public:
    void check_invariants() const {
        for (std::size_t i = 1; i < m_tree.size(); ++i) {
            CONTRACT_INVARIANT(m_tree[(i - 1) / 2] <= m_tree[i]);
        }
    }
    [[nodiscard]] const T& peek() const {
        CONTRACT_EXPECTS(!m_tree.empty());
        return m_tree.front();
    }
    [[nodiscard]] std::size_t size() const noexcept { return m_tree.size(); }
};
```

**Verification Tasks:**

- **Task 1 (Heap Invariant Maintenance):** Insert 1,000 random integers
  into `ContractedMinHeap`. Verify that `check_invariants()` passes
  after every individual insertion.
- **Task 2 (Heapsort Property):** Pop all elements sequentially from the
  heap. Verify that the resulting output stream is sorted in ascending
  order.
- **Task 3 (Invariant Violation Detection):** Manually swap elements at
  index `0` and index `m_tree.size() - 1` using pointer manipulation.
  Verify that calling `check_invariants()` catches the ordering
  violation.

#### Exercise 15.9: Termination Proof for Euclid’s Greatest Common Divisor (GCD)

**Context and Theory:**

Euclid’s algorithm for computing the Greatest Common Divisor
($\gcd(a, b)$) of two integers is one of the oldest numerical
algorithms:

$$\gcd(a, b) = \begin{cases} a & \text{if } b = 0 \\ \gcd(b, a \pmod b) & \text{if } b > 0 \end{cases}$$

To prove that Euclid’s algorithm is correct, we establish two theorems:

1.  **Partial Correctness (Invariant Preservation):** If
    $a = q \cdot b + r$ where $r = a \pmod b$, then any common divisor
    of $a$ and $b$ must also divide $r$. Hence:
    $$\gcd(a, b) \equiv \gcd(b, a \pmod b)$$
2.  **Total Termination (Variant Function Proof):** We define the
    ranking function $V(a, b) = b \in \mathbb{N}_0$. In each step
    $(a_{k+1}, b_{k+1}) = (b_k, a_k \pmod{b_k})$. By definition of the
    modulo operator:
    $$0 \le a_k \pmod{b_k} < b_k \implies b_{k+1} < b_k$$ The sequence
    of second arguments $b_0, b_1, b_2, \dots$ forms a strictly
    decreasing sequence of non-negative integers. By the **Well-Ordering
    Principle of Natural Numbers**, the algorithm must reach $b = 0$ in
    a finite number of steps.

**Specification:**

Implement `contracted_gcd`:

1.  Check precondition: $a \ge 0$ and $b \ge 0$.
2.  Implement the loop, verifying at each step that:
    - The variant $V_k = b$ strictly decreases ($b_{k+1} < b_k$).
    - A known divisor invariant is preserved.
3.  Check postcondition: the returned value divides both $a$ and $b$
    with zero remainder.

``` cpp
#include <cstdint>
// #include "Contracts.hpp"

[[nodiscard]] std::uint64_t contracted_gcd(std::uint64_t a, std::uint64_t b) {
    CONTRACT_EXPECTS(a > 0 || b > 0);
    const std::uint64_t orig_a = a;
    const std::uint64_t orig_b = b;

    while (b != 0) {
        const std::uint64_t prev_b = b; // Variant checkpoint
        const std::uint64_t rem = a % b;
        a = b;
        b = rem;
        CONTRACT_INVARIANT(b < prev_b); // Variant strictly decreases!
    }

    CONTRACT_ENSURES(orig_a % a == 0); // Divides a
    CONTRACT_ENSURES(orig_b % a == 0); // Divides b
    return a;
}
```

**Verification Tasks:**

- **Task 1 (Coprime Convergence):** Compute
  `contracted_gcd(104729, 104723)` (two large prime numbers). Verify
  that the function terminates, all postconditions pass, and the result
  is `1`.
- **Task 2 (Fibonacci Worst-Case Termination):** Consecutive Fibonacci
  numbers represent the worst-case input for Euclid’s algorithm (taking
  the maximum number of steps). Compute `contracted_gcd(F(40), F(39))`.
  Verify that the variant strictly decreases on every iteration.
- **Task 3 (Zero Precondition Handling):** Call `contracted_gcd(0, 0)`.
  Verify that the precondition assertion fires and halts execution.

#### Exercise 15.10: Zero-Overhead Contract Audit Levels via `[[assume]]` and Attributes

**Context and Theory:**

While formal contract assertions catch bugs during development, checking
complex postconditions and class invariants (such as traversing an
entire tree or scanning an array to verify it is sorted) introduces
significant runtime overhead.

A production-grade contract architecture supports **Configurable
Contract Levels**:

1.  **Audit Level (`CONTRACT_LEVEL_AUDIT`):** Checks all preconditions,
    postconditions, and expensive class invariants. Used during
    development and CI testing.
2.  **Default Level (`CONTRACT_LEVEL_DEFAULT`):** Checks only
    lightweight preconditions ($O(1)$ pointer checks).
3.  **Release Level (`CONTRACT_LEVEL_RELEASE`):** Contracts are compiled
    away for zero runtime overhead.

In C++23, compiler hints can be supplied using **`[[assume(expr)]]`**.

When contracts are disabled in release builds, instead of discarding the
predicate, the macro expands to `[[assume(expr)]]`. This tells the
compiler’s optimizer that the condition is guaranteed to be true,
enabling optimisations such as eliminating dead branches, eliding bounds
checks, and vectorising loops.

``` cpp
// Debug Build: Emits runtime check
if (!cond) throw ContractViolation();

// Release Build: Emits zero instructions, informs optimizer
[[assume(cond)]];
```

**Specification:**

Design a multi-tiered contract header `ContractLevels.hpp`: 1. Support
preprocessor configurations: `LEVEL_AUDIT`, `LEVEL_DEFAULT`, and
`LEVEL_RELEASE`. 2. Under `LEVEL_AUDIT`, evaluate all contracts and
throw diagnostic exceptions on failure. 3. Under `LEVEL_RELEASE`,
translate contracts to C++23 `[[assume(cond)]]` (or compiler built-ins
`__builtin_assume` on Clang / `__assume` on MSVC). 4. Verify on Compiler
Explorer that a function containing a range check optimizes away dead
branches when compiled under `LEVEL_RELEASE`.

``` cpp
#if defined(CONTRACT_LEVEL_AUDIT)
    #define CONTRACT_CHECK(cond) proof::contract_assert((cond), "Audit", #cond)
#elif defined(CONTRACT_LEVEL_RELEASE)
    #if defined(__clang__)
        #define CONTRACT_CHECK(cond) __builtin_assume(cond)
    #elif defined(_MSC_VER)
        #define CONTRACT_CHECK(cond) __assume(cond)
    #else
        #define CONTRACT_CHECK(cond) [[assume(cond)]]
    #endif
#else
    #define CONTRACT_CHECK(cond) do {} while(false)
#endif
```

**Verification Tasks:**

- **Task 1 (Audit Catch Verification):** Compile a test case with
  `CONTRACT_LEVEL_AUDIT`. Trigger a contract violation. Verify that a
  detailed diagnostic exception is thrown.

- **Task 2 (Release Assembly Invariant Elision):** Write a function:

  ``` cpp
  int get_value(int idx) {
      CONTRACT_CHECK(idx >= 0 && idx < 4);
      if (idx < 0) return -1; // Dead branch!
      return data[idx];
  }
  ```

  Compile under `CONTRACT_LEVEL_RELEASE` with `-O3`. Verify on Compiler
  Explorer that the `if (idx < 0)` branch is completely eliminated from
  the emitted assembly.

- **Task 3 (Zero Performance Overhead):** Benchmark 10,000,000 function
  calls under `CONTRACT_LEVEL_RELEASE`. Verify that the execution time
  matches an uninstrumented baseline.

#### Exercise 15.11: Symbolic Execution Path Constraint Solver Simulation

**Context and Theory:**

Empirical testing evaluates code using concrete inputs ($x = 5$).
**Symbolic Execution** evaluates programs using **Symbolic Variables**
($\alpha$).

As the symbolic engine interprets code, it tracks:

1.  **Symbolic State:** Variables mapped to symbolic expressions (e.g.,
    $y \leftarrow 2\alpha + 1$).
2.  **Path Condition ($\text{PC}$):** A boolean formula collecting the
    branch decisions taken along the current execution path.

When the interpreter encounters a conditional statement `if (x > 10)`:

- The execution path branches into two independent execution states:
  - **Then-Branch:**
    $\text{PC}_{\text{true}} = \text{PC} \land (\alpha > 10)$
  - **Else-Branch:**
    $\text{PC}_{\text{false}} = \text{PC} \land \neg(\alpha > 10)$

A **Satisfiability Modulo Theories (SMT) Solver** (such as Z3) inspects
the path conditions (see fig. 7):

- If a path condition evaluates to **UNSAT** (Unsatisfiable), that path
  represents **Dead Code** (it can never execute under any possible
  input).
- If **SAT** (Satisfiable), the solver generates a concrete input that
  forces execution down that exact path, achieving 100% path coverage.

<figure id="fig:sat-unsat-path">
<img src="data:image/svg+xml;base64,PHN2ZyB2aWV3Qm94PSI0MCAwIDg2MCAxNTAiIHdpZHRoPSIxMDAlIiBoZWlnaHQ9IjE1MCIgeG1s
bnM9Imh0dHA6Ly93d3cudzMub3JnLzIwMDAvc3ZnIj4KICA8ZGVmcz4KICAgIDxzdHlsZT4KICAg
ICAgOnJvb3QgewogICAgICAgIC0tc3ZnLWJnOiB2YXIoLS1iZywgI2ZmZmZmZik7CiAgICAgICAg
LS1zdmctZmc6IHZhcigtLWZnLCAjMDAwMDAwKTsKICAgICAgICAtLXN2Zy10aW50LTEtZmFsbGJh
Y2s6ICNjY2NjY2M7IC8qIDIwJSBibGFjayArIDgwJSB3aGl0ZSAqLwogICAgICAgIC0tc3ZnLXRp
bnQtMi1mYWxsYmFjazogIzk5OTk5OTsgLyogNDAlIGJsYWNrICsgNjAlIHdoaXRlICovCiAgICAg
ICAgLS1zdmctdGludC0xOiBjb2xvci1taXgoaW4gc3JnYiwgdmFyKC0tc3ZnLWZnKSAyMCUsIHZh
cigtLXN2Zy1iZykgODAlKTsKICAgICAgICAtLXN2Zy10aW50LTI6IGNvbG9yLW1peChpbiBzcmdi
LCB2YXIoLS1zdmctZmcpIDQwJSwgdmFyKC0tc3ZnLWJnKSA2MCUpOwogICAgICB9CiAgICAgIHRl
eHQgICB7IGZvbnQtZmFtaWx5OiAiSW9zZXZrYSBDaGFyb24gTW9ubyBNZWRpdW0iLCBtb25vc3Bh
Y2U7IH0KICAgIDwvc3R5bGU+CiAgPC9kZWZzPgogIDxyZWN0IHg9IjUwIiB5PSIzMCIgd2lkdGg9
IjIyMCIgaGVpZ2h0PSI1MCIgcng9IjYiIGZpbGw9InZhcigtLXN2Zy10aW50LTEsIHZhcigtLXN2
Zy10aW50LTEtZmFsbGJhY2spKSIgc3Ryb2tlPSJ2YXIoLS1zdmctZmcpIi8+CiAgPHRleHQgeD0i
NjUiIHk9IjU1IiBmb250LXNpemU9IjE0IiBmaWxsPSJ2YXIoLS1zdmctZmcpIj5Sb290OiBQQyA9
IFRydWUgW3ggPSAmIzk0NTtdPC90ZXh0PgogIDxyZWN0IHg9IjM0MCIgeT0iMTAiIHdpZHRoPSIy
MzAiIGhlaWdodD0iNTAiIHJ4PSI2IiBmaWxsPSJ2YXIoLS1zdmctdGludC0xLCB2YXIoLS1zdmct
dGludC0xLWZhbGxiYWNrKSkiIHN0cm9rZT0idmFyKC0tc3ZnLWZnKSIvPgogIDx0ZXh0IHg9IjM1
NSIgeT0iMzUiIGZvbnQtc2l6ZT0iMTQiIGZpbGw9InZhcigtLXN2Zy1mZykiPkJyYW5jaCAxOiAm
Izk0NTsgJmd0OyAxMDwvdGV4dD4KICA8dGV4dCB4PSIzNTUiIHk9IjUwIiBmb250LXNpemU9IjE0
IiBmaWxsPSJ2YXIoLS1zdmctZmcpIj5TQVQgJiM4NTk0OyBDb25jcmV0ZSBJbnB1dDogeCA9IDEx
PC90ZXh0PgogIDxyZWN0IHg9IjM0MCIgeT0iODAiIHdpZHRoPSIyMzAiIGhlaWdodD0iNTAiIHJ4
PSI2IiBmaWxsPSJ2YXIoLS1zdmctdGludC0xLCB2YXIoLS1zdmctdGludC0xLWZhbGxiYWNrKSki
IHN0cm9rZT0idmFyKC0tc3ZnLWZnKSIvPgogIDx0ZXh0IHg9IjM1NSIgeT0iMTA1IiBmb250LXNp
emU9IjE0IiBmaWxsPSJ2YXIoLS1zdmctZmcpIj5CcmFuY2ggMjogJiM5NDU7ICYjODgwNDsgMTA8
L3RleHQ+CiAgPHRleHQgeD0iMzU1IiB5PSIxMjAiIGZvbnQtc2l6ZT0iMTQiIGZpbGw9InZhcigt
LXN2Zy1mZykiPlNBVCAmIzg1OTQ7IENvbmNyZXRlIElucHV0OiB4ID0gMDwvdGV4dD4KICA8cmVj
dCB4PSI2MjAiIHk9IjEwIiB3aWR0aD0iMjIwIiBoZWlnaHQ9IjUwIiByeD0iNiIgZmlsbD0idmFy
KC0tc3ZnLXRpbnQtMSwgdmFyKC0tc3ZnLXRpbnQtMS1mYWxsYmFjaykpIiBzdHJva2U9InZhcigt
LXN2Zy1mZykiIHN0cm9rZS1kYXNoYXJyYXk9IjMsMyIvPgogIDx0ZXh0IHg9IjYyNSIgeT0iMzUi
IGZvbnQtc2l6ZT0iMTQiIGZpbGw9InZhcigtLXN2Zy1mZykiPk5lc3RlZDogKCYjOTQ1OyAmZ3Q7
IDEwKSAmYW1wOyAoJiM5NDU7ICZsdDsgNSk8L3RleHQ+CiAgPHRleHQgeD0iNjI1IiB5PSI1MCIg
Zm9udC1zaXplPSIxNCIgZmlsbD0idmFyKC0tc3ZnLWZnKSI+VU5TQVQhIChEZWFkIENvZGUpPC90
ZXh0PgogIDxwYXRoIGQ9Ik0gMjcwIDU1IEwgMzQwIDM1IiBzdHJva2U9InZhcigtLXN2Zy1mZyki
IHN0cm9rZS13aWR0aD0iMiIvPgogIDxwYXRoIGQ9Ik0gMjcwIDU1IEwgMzQwIDEwNSIgc3Ryb2tl
PSJ2YXIoLS1zdmctZmcpIiBzdHJva2Utd2lkdGg9IjIiLz4KICA8cGF0aCBkPSJNIDU3MCAzNSBM
IDYyMCAzNSIgc3Ryb2tlPSJ2YXIoLS1zdmctZmcpIiBzdHJva2Utd2lkdGg9IjIiLz4KPC9zdmc+
" alt="Figure 7: SAT and UNSAT Paths" />
<figcaption>Figure 7: SAT and UNSAT Paths</figcaption>
</figure>

**Specification:**

Design a mini symbolic path condition tracer:

1.  Define a symbolic variable representation: `SymbolicInt{string id}`.
2.  Implement relational expressions (`SymExpr`) capable of forming
    constraint trees: `(alpha > 10) && (alpha < 50)`.
3.  Implement a simple constraint solver that checks interval
    feasibility for bounded integer ranges $[L, R]$.
4.  Trace a function with multiple branches, printing the path
    conditions required to hit each branch and identifying unreachable
    blocks.

``` cpp
#include <string>
#include <vector>
#include <iostream>

struct IntervalConstraint {
    int min_val{-1000000};
    int max_val{1000000};

    bool add_greater_than(int val) {
        min_val = std::max(min_val, val + 1);
        return min_val <= max_val;
    }
    bool add_less_than(int val) {
        max_val = std::min(max_val, val - 1);
        return min_val <= max_val;
    }
    [[nodiscard]] bool is_satisfiable() const noexcept { return min_val <= max_val; }
    [[nodiscard]] int get_concrete_solution() const { return min_val; }
};
```

**Verification Tasks:**

- **Task 1 (Reachable Path Solution):** Trace
  `if (x > 5) { if (x < 10) { ... } }`. Verify that the constraint
  tracker computes the interval $[6, 9]$ and yields a valid concrete
  input (e.g., $x = 6$).
- **Task 2 (Dead Code Detection):** Trace
  `if (x > 10) { if (x < 5) { UNREACHABLE(); } }`. Verify that the
  constraint solver identifies that the interval constraint is
  unsatisfiable (`is_satisfiable() == false`), proving the nested block
  is dead code.
- **Task 3 (Full Branch Coverage Generation):** Provide a three-way
  conditional branch. Verify that the tracer generates a set of concrete
  inputs that execute every branch.

#### Exercise 15.12: Inductive Correctness Proof of Red-Black Tree Colour Invariants

**Context and Theory:**

A **Red-Black Tree** is an approximately balanced binary search tree
that satisfies five structural invariants:

1.  **Node Colour:** Every node is either **Red** or **Black**.
2.  **Root Property:** The root node is always **Black**.
3.  **Leaf Property:** Every leaf node (`NIL` sentinel) is **Black**.
4.  **Red Property:** If a node is **Red**, both of its children must be
    **Black** (no two consecutive red nodes on any path).
5.  **Black-Height Invariant:** Every simple path from a given node to
    any of its descendant `NIL` leaves contains the **exact same number
    of black nodes**:
    $$\text{BlackHeight}(N) = k \quad \forall \text{ paths from } N \text{ to leaves}$$

<figure id="fig:red-black-tree">
<img src="data:image/svg+xml;base64,PD94bWwgdmVyc2lvbj0iMS4wIiBlbmNvZGluZz0iVVRGLTgiIHN0YW5kYWxvbmU9Im5vIj8+Cjwh
RE9DVFlQRSBzdmcgUFVCTElDICItLy9XM0MvL0RURCBTVkcgMS4xLy9FTiIKICJodHRwOi8vd3d3
LnczLm9yZy9HcmFwaGljcy9TVkcvMS4xL0RURC9zdmcxMS5kdGQiPgo8IS0tIEdlbmVyYXRlZCBi
eSBncmFwaHZpeiB2ZXJzaW9uIDIuNDMuMCAoMCkKIC0tPgo8IS0tIFRpdGxlOiBHIFBhZ2VzOiAx
IC0tPgo8c3ZnIHdpZHRoPSIyNzNwdCIgaGVpZ2h0PSIxMDdwdCIKIHZpZXdCb3g9IjAuMDAgMC4w
MCAyNzMuMDAgMTA3LjAwIiB4bWxucz0iaHR0cDovL3d3dy53My5vcmcvMjAwMC9zdmciIHhtbG5z
OnhsaW5rPSJodHRwOi8vd3d3LnczLm9yZy8xOTk5L3hsaW5rIj4KPGcgaWQ9ImdyYXBoMCIgY2xh
c3M9ImdyYXBoIiB0cmFuc2Zvcm09InNjYWxlKDEgMSkgcm90YXRlKDApIHRyYW5zbGF0ZSg0IDEw
MykiPgo8dGl0bGU+RzwvdGl0bGU+CjwhLS0gMTMgLS0+CjxnIGlkPSJub2RlMSIgY2xhc3M9Im5v
ZGUiPgo8dGl0bGU+MTM8L3RpdGxlPgo8ZWxsaXBzZSBmaWxsPSJub25lIiBzdHJva2U9ImJsYWNr
IiBjeD0iMTEyLjUiIGN5PSItNzciIHJ4PSIyMS41IiByeT0iMTEiLz4KPHRleHQgdGV4dC1hbmNo
b3I9Im1pZGRsZSIgeD0iMTEyLjUiIHk9Ii03NC41IiBmb250LWZhbWlseT0iSW9zZXZrYSBDaGFy
b24gTW9ubyBNZWRpdW0iIGZvbnQtc2l6ZT0iMTAuMDAiPjEzIChCKTwvdGV4dD4KPHRleHQgdGV4
dC1hbmNob3I9Im1pZGRsZSIgeD0iNTMuNSIgeT0iLTkxIiBmb250LWZhbWlseT0iSW9zZXZrYSBD
aGFyb24gTW9ubyBNZWRpdW0iIGZvbnQtc2l6ZT0iMTAuMDAiPkJsYWNrIEhlaWdodCA9IDI8L3Rl
eHQ+CjwvZz4KPCEtLSA4IC0tPgo8ZyBpZD0ibm9kZTIiIGNsYXNzPSJub2RlIj4KPHRpdGxlPjg8
L3RpdGxlPgo8ZWxsaXBzZSBmaWxsPSJub25lIiBzdHJva2U9ImJsYWNrIiBjeD0iODIuNSIgY3k9
Ii00NCIgcng9IjIxLjUiIHJ5PSIxMSIvPgo8dGV4dCB0ZXh0LWFuY2hvcj0ibWlkZGxlIiB4PSI4
Mi41IiB5PSItNDEuNSIgZm9udC1mYW1pbHk9Iklvc2V2a2EgQ2hhcm9uIE1vbm8gTWVkaXVtIiBm
b250LXNpemU9IjEwLjAwIj44IChSKTwvdGV4dD4KPC9nPgo8IS0tIDEzJiM0NTsmZ3Q7OCAtLT4K
PGcgaWQ9ImVkZ2UxIiBjbGFzcz0iZWRnZSI+Cjx0aXRsZT4xMyYjNDU7Jmd0Ozg8L3RpdGxlPgo8
cGF0aCBmaWxsPSJub25lIiBzdHJva2U9ImJsYWNrIiBkPSJNMTAzLjY2LC02Ni44N0M5OS43OSwt
NjIuODcgOTUuMjYsLTU4LjE4IDkxLjM4LC01NC4xOCIvPgo8L2c+CjwhLS0gMTcgLS0+CjxnIGlk
PSJub2RlMyIgY2xhc3M9Im5vZGUiPgo8dGl0bGU+MTc8L3RpdGxlPgo8ZWxsaXBzZSBmaWxsPSJu
b25lIiBzdHJva2U9ImJsYWNrIiBjeD0iMTQzLjUiIGN5PSItNDQiIHJ4PSIyMS41IiByeT0iMTEi
Lz4KPHRleHQgdGV4dC1hbmNob3I9Im1pZGRsZSIgeD0iMTQzLjUiIHk9Ii00MS41IiBmb250LWZh
bWlseT0iSW9zZXZrYSBDaGFyb24gTW9ubyBNZWRpdW0iIGZvbnQtc2l6ZT0iMTAuMDAiPjE3IChC
KTwvdGV4dD4KPHRleHQgdGV4dC1hbmNob3I9InN0YXJ0IiB4PSIxNjUiIHk9Ii02OSIgZm9udC1m
YW1pbHk9Iklvc2V2a2EgQ2hhcm9uIE1vbm8gTWVkaXVtIiBmb250LXNpemU9IjEwLjAwIj5QYXRo
cyB0byBOSUwgaGF2ZTwvdGV4dD4KPHRleHQgdGV4dC1hbmNob3I9InN0YXJ0IiB4PSIxNjUiIHk9
Ii01OCIgZm9udC1mYW1pbHk9Iklvc2V2a2EgQ2hhcm9uIE1vbm8gTWVkaXVtIiBmb250LXNpemU9
IjEwLjAwIj5leGFjdGx5IDIgQmxhY2sgbm9kZXMhPC90ZXh0Pgo8L2c+CjwhLS0gMTMmIzQ1OyZn
dDsxNyAtLT4KPGcgaWQ9ImVkZ2UyIiBjbGFzcz0iZWRnZSI+Cjx0aXRsZT4xMyYjNDU7Jmd0OzE3
PC90aXRsZT4KPHBhdGggZmlsbD0ibm9uZSIgc3Ryb2tlPSJibGFjayIgZD0iTTEyMS42MywtNjYu
ODdDMTI1LjYzLC02Mi44NyAxMzAuMzIsLTU4LjE4IDEzNC4zMiwtNTQuMTgiLz4KPC9nPgo8IS0t
IDEgLS0+CjxnIGlkPSJub2RlNCIgY2xhc3M9Im5vZGUiPgo8dGl0bGU+MTwvdGl0bGU+CjxlbGxp
cHNlIGZpbGw9Im5vbmUiIHN0cm9rZT0iYmxhY2siIGN4PSIyMS41IiBjeT0iLTExIiByeD0iMjEu
NSIgcnk9IjExIi8+Cjx0ZXh0IHRleHQtYW5jaG9yPSJtaWRkbGUiIHg9IjIxLjUiIHk9Ii04LjUi
IGZvbnQtZmFtaWx5PSJJb3NldmthIENoYXJvbiBNb25vIE1lZGl1bSIgZm9udC1zaXplPSIxMC4w
MCI+MSAoQik8L3RleHQ+CjwvZz4KPCEtLSA4JiM0NTsmZ3Q7MSAtLT4KPGcgaWQ9ImVkZ2UzIiBj
bGFzcz0iZWRnZSI+Cjx0aXRsZT44JiM0NTsmZ3Q7MTwvdGl0bGU+CjxwYXRoIGZpbGw9Im5vbmUi
IHN0cm9rZT0iYmxhY2siIGQ9Ik02OC4wNCwtMzUuNjVDNTguMzIsLTMwLjcxIDQ1LjU1LC0yNC4y
MiAzNS44NSwtMTkuMjkiLz4KPC9nPgo8IS0tIDExIC0tPgo8ZyBpZD0ibm9kZTUiIGNsYXNzPSJu
b2RlIj4KPHRpdGxlPjExPC90aXRsZT4KPGVsbGlwc2UgZmlsbD0ibm9uZSIgc3Ryb2tlPSJibGFj
ayIgY3g9IjgyLjUiIGN5PSItMTEiIHJ4PSIyMS41IiByeT0iMTEiLz4KPHRleHQgdGV4dC1hbmNo
b3I9Im1pZGRsZSIgeD0iODIuNSIgeT0iLTguNSIgZm9udC1mYW1pbHk9Iklvc2V2a2EgQ2hhcm9u
IE1vbm8gTWVkaXVtIiBmb250LXNpemU9IjEwLjAwIj4xMSAoQik8L3RleHQ+CjwvZz4KPCEtLSA4
JiM0NTsmZ3Q7MTEgLS0+CjxnIGlkPSJlZGdlNCIgY2xhc3M9ImVkZ2UiPgo8dGl0bGU+OCYjNDU7
Jmd0OzExPC90aXRsZT4KPHBhdGggZmlsbD0ibm9uZSIgc3Ryb2tlPSJibGFjayIgZD0iTTgyLjUs
LTMyLjg1QzgyLjUsLTI5LjQ2IDgyLjUsLTI1LjY4IDgyLjUsLTIyLjI4Ii8+CjwvZz4KPCEtLSAy
NSAtLT4KPGcgaWQ9Im5vZGU2IiBjbGFzcz0ibm9kZSI+Cjx0aXRsZT4yNTwvdGl0bGU+CjxlbGxp
cHNlIGZpbGw9Im5vbmUiIHN0cm9rZT0iYmxhY2siIGN4PSIxNDMuNSIgY3k9Ii0xMSIgcng9IjIx
LjUiIHJ5PSIxMSIvPgo8dGV4dCB0ZXh0LWFuY2hvcj0ibWlkZGxlIiB4PSIxNDMuNSIgeT0iLTgu
NSIgZm9udC1mYW1pbHk9Iklvc2V2a2EgQ2hhcm9uIE1vbm8gTWVkaXVtIiBmb250LXNpemU9IjEw
LjAwIj4yNSAoUik8L3RleHQ+CjwvZz4KPCEtLSAxNyYjNDU7Jmd0OzI1IC0tPgo8ZyBpZD0iZWRn
ZTUiIGNsYXNzPSJlZGdlIj4KPHRpdGxlPjE3JiM0NTsmZ3Q7MjU8L3RpdGxlPgo8cGF0aCBmaWxs
PSJub25lIiBzdHJva2U9ImJsYWNrIiBkPSJNMTQzLjUsLTMyLjg1QzE0My41LC0yOS40NiAxNDMu
NSwtMjUuNjggMTQzLjUsLTIyLjI4Ii8+CjwvZz4KPC9nPgo8L3N2Zz4K
" alt="" />
<figcaption>Figure 8: Black-Height Verification</figcaption>
</figure>

**Mathematical Induction on Tree Insertion:**

To prove that a Red-Black tree insertion preserves balance:

- By Invariant 4, the longest possible path from the root to a leaf
  alternates between Red and Black nodes
  ($\text{Length} \le 2 \times \text{BlackHeight}$).
- By Invariant 5, the shortest possible path contains only Black nodes
  ($\text{Length} = \text{BlackHeight}$).
- Therefore, the longest path is at most twice the length of the
  shortest path: $$\text{Height} \le 2 \log_2(N + 1)$$ The tree is
  guaranteed to maintain $O(\log N)$ height.

**Specification:**

Implement an invariant verification suite for a `RedBlackTree<T>`:

1.  Define `enum class Colour { Red, Black };`.
2.  Implement `check_rb_invariants()`:
    - Verify Root is Black:
      `CONTRACT_INVARIANT(m_root->colour == Colour::Black);`

    - Verify Red Property (no red child under a red parent):

      ``` cpp
      if (node->colour == Colour::Red) {
          CONTRACT_INVARIANT(get_colour(node->left) == Colour::Black);
          CONTRACT_INVARIANT(get_colour(node->right) == Colour::Black);
      }
      ```

    - Verify Black-Height Invariant: recursively compute the
      black-height for every path to a leaf and assert they are
      identical.
3.  Assert these invariants after every insert and delete operation.

``` cpp
// #include "Contracts.hpp"

enum class NodeColour : bool { Red, Black };

template <typename T>
class VerifiedRedBlackTree {
    struct Node {
        T key;
        NodeColour colour{NodeColour::Red};
        Node* left{nullptr};
        Node* right{nullptr};
    };
    Node* m_root{nullptr};

    static int verify_black_height(const Node* n) {
        if (!n) return 1; // NIL leaf is Black
        const int left_bh = verify_black_height(n->left);
        const int right_bh = verify_black_height(n->right);
        CONTRACT_INVARIANT(left_bh == right_bh); // Black-Height must match!
        return left_bh + (n->colour == NodeColour::Black ? 1 : 0);
    }
public:
    void check_invariants() const {
        if (!m_root) return;
        CONTRACT_INVARIANT(m_root->colour == NodeColour::Black);
        verify_black_height(m_root);
    }
};
```

**Verification Tasks:**

- **Task 1 (Sequential Insertion Balancing):** Insert 100 consecutive
  ascending integers. Verify that the re-colouring and rotation passes
  maintain identical black-heights across all paths.
- **Task 2 (Consecutive Red Violation Catch):** Deliberately alter the
  colour flip logic so that an inserted node’s parent remains Red.
  Verify that `CONTRACT_INVARIANT` catches the consecutive-red
  violation.
- **Task 3 (Black-Height Desync Detection):** Force an unbalanced leaf
  insertion that increases the black count of only one subtree. Verify
  that `verify_black_height` triggers an assertion failure.